# D3-01 · Segmentation basics: tasks, metrics, losses, upsampling

### Steps
1. Semantic vs instance vs panoptic segmentation on one Pet image; label formats and RLE
2. Metrics: why pixel accuracy lies; confusion matrix -> IoU -> mIoU; Dice
3. Losses: cross-entropy / BCE, soft Dice, focal; who gets the gradient on a small object
4. Getting resolution back: transposed convolution, the checkerboard, bilinear + conv, pixel shuffle
5. Context without downsampling: dilated (atrous) convolutions and ASPP
6. The autoencoder ancestor: what a bottleneck forgets
7. Build the Pet trimap target and look at the class frequencies

### Topics covered and learning objectives
- The three segmentation tasks and how their labels are stored (masks, trimaps, polygons, RLE)
- mIoU and Dice from a confusion matrix; Dice = 2 IoU / (1 + IoU)
- Why Dice / focal losses help small objects, measured with autograd
- Transposed convolution = scatter-add; why stride 2 / kernel 3 makes a checkerboard
- Pixel shuffle, dilation, ASPP: the toolbox every segmentation decoder is built from
- Encoder-decoder without skips loses 'where' - the motivation for U-Net (next notebook)

### Time estimate
~90 min in class (guided + hands-on)

### Prerequisites
Day 1 (convolutions, transfer learning) and Day 2 (detection, RoIAlign, FPN).

## 0 · Setup

NumPy for the mechanisms, PyTorch for the losses, the upsampling layers and one small
autoencoder. Everything runs on a CPU in a few minutes.

<details class="api-ref" data-api="np.set_printoptions,torch.manual_seed,torchkit.get_device"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.set_printoptions</code> · <code>torch.manual_seed</code> · <code>torchkit.get_device</code></summary>
<ul><li><code>np.set_printoptions(precision=2, suppress=True)</code> &nbsp; Changes how NumPy prints arrays for the rest of the session: precision=2 shows two decimals, suppress=True writes tiny numbers as 0. instead of in scientific notation (2e-05). Only the display changes, never the values.</li><li><code>torch.manual_seed(seed)</code> &nbsp; Fixes PyTorch&#x27;s random number generator, so random weights, shuffles and augmentations come out the same on every run. The notebooks call it in the setup cell and again right before creating a model, so that its initial weights are repeatable.</li><li><code>device = torchkit.get_device()</code> &nbsp; Course helper. Returns torch.device(&quot;cuda&quot;) when PyTorch can use an NVIDIA GPU, torch.device(&quot;mps&quot;) on an Apple-silicon Mac, otherwise torch.device(&quot;cpu&quot;), and prints which one it picked (for the CPU, with the number of threads PyTorch uses). Models and batches are then moved there with .to(device).</li></ul></details>

In [ ]:
import sys, pathlib
for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:   # find the cvcourse package
    if (p / "cvcourse").is_dir():
        sys.path.insert(0, str(p)); break

import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Patch
import torch, torch.nn as nn, torch.nn.functional as F
from cvcourse import viz, convops, seg, shapes, tests, torchkit, day3, labs

%matplotlib inline
plt.rcParams["figure.dpi"] = 90
np.set_printoptions(precision=2, suppress=True)
torch.manual_seed(0)
device = torchkit.get_device()
if device.type == "cpu":
    day3.tune_cpu_threads()

## 1 · Three segmentation tasks

Detection (Day 2) answers *"what is where, roughly?"* with boxes. Segmentation answers
the same question **per pixel**:

| task | output per pixel | two cats of the same breed | counts objects? |
|---|---|---|---|
| **semantic** | a class | one blob labelled "cat" | no |
| **instance** | a class + an instance id, *things only* | "cat #1", "cat #2" | yes |
| **panoptic** | class + id for *things*, class only for *stuff* (sky, road, grass) | "cat #1", "cat #2", "background" | yes |

(*Things* are countable objects such as cats, cars, people; *stuff* is amorphous background such as sky, road or grass.)

The Oxford-IIIT Pet image below happens to contain **two** Bengal cats. The dataset gives
a *trimap* (pet / background / uncertain border) and one head box per animal.

<details class="api-ref" data-api="scipy.ndimage.label,scipy.ndimage.binary_opening,np.unique,np.dstack"><summary><b>Functions used here</b> (click to open): <code>scipy.ndimage.label</code> · <code>scipy.ndimage.binary_opening</code> · <code>np.unique</code> · <code>np.dstack</code></summary>
<p><b><code>labels, n = ndimage.label(mask)</code></b><br>Finds the connected regions of a binary mask and numbers them. Returns an integer array of the mask&#x27;s shape (0 = background, 1..n = one number per region) and the number of regions n. Pixels are connected through their 4 side neighbours, not through diagonals. D3-01 uses it to split a pet mask into one region per cat.</p><pre>&gt;&gt;&gt; from scipy import ndimage
&gt;&gt;&gt; m = np.array([[1, 1, 0, 0],
...               [0, 0, 0, 1],
...               [1, 0, 1, 1]])
&gt;&gt;&gt; labels, n = ndimage.label(m)
&gt;&gt;&gt; n
3
&gt;&gt;&gt; labels
array([[1, 1, 0, 0],
       [0, 0, 0, 2],
       [3, 0, 2, 2]], dtype=int32)</pre><p><i>Watch out:</i> Touching objects become one region: connected components separate instances only when they do not touch (two people side by side need Mask R-CNN, D3-03). structure=np.ones((3, 3)) also connects diagonal neighbours.</p><p>See also: <code>scipy.ndimage.binary_opening</code>, <code>np.unique</code></p><p><b><code>ndimage.binary_opening(mask, iterations=1)</code></b><br>Removes small specks and thin strands from a binary mask: it first shrinks every True region (erosion, see binary_erosion), then grows what is left back by the same amount. Parts narrower than about 2 * iterations + 1 pixels disappear; larger regions keep their shape, with slightly rounded corners. D3-01 opens the pet mask before ndimage.label so that stray pixels do not count as extra instances.</p><pre>&gt;&gt;&gt; from scipy import ndimage
&gt;&gt;&gt; m = np.zeros((8, 8), bool)
&gt;&gt;&gt; m[2:6, 1:5] = True; m[0, 7] = True           # a 4x4 blob and one stray pixel
&gt;&gt;&gt; ndimage.label(m)[1]                          # number of regions
2
&gt;&gt;&gt; ndimage.label(ndimage.binary_opening(m))[1]  # the stray pixel is gone
1</pre><p>See also: <code>scipy.ndimage.binary_erosion</code>, <code>scipy.ndimage.label</code></p><p><b><code>np.unique(a, return_counts=False)</code></b><br>The distinct values of an array, sorted (the array is flattened first). The quickest check of what a label mask contains: a raw Pet trimap gives [1 2 3]. With return_counts=True it also returns how often each value occurs, e.g. the number of pixels per class.</p><pre>&gt;&gt;&gt; mask = np.array([[0, 0, 2],
...                  [1, 2, 2]])
&gt;&gt;&gt; np.unique(mask)
array([0, 1, 2])
&gt;&gt;&gt; values, counts = np.unique(mask, return_counts=True)
&gt;&gt;&gt; counts                                       # two 0s, one 1, three 2s
array([2, 1, 3])</pre><p>See also: <code>torch.unique</code>, <code>np.bincount</code></p><p><b><code>np.dstack([a, b, ...])</code></b><br>Stacks arrays along a third axis (&quot;depth&quot;): two (H, W) arrays give (H, W, 2); an (H, W, 3) RGB image plus an (H, W) array give (H, W, 4). D3-01 uses it to add an alpha (transparency) channel to a colour overlay, because plt.imshow draws an (H, W, 4) array as RGBA.</p><pre>&gt;&gt;&gt; rgb = np.zeros((2, 3, 3))                    # H=2, W=3, RGB
&gt;&gt;&gt; alpha = np.ones((2, 3))                      # one opacity value per pixel
&gt;&gt;&gt; np.dstack([rgb, alpha]).shape                # RGBA
(2, 3, 4)
&gt;&gt;&gt; np.dstack([np.zeros((2, 3)), np.ones((2, 3))]).shape
(2, 3, 2)</pre><p>See also: <code>np.stack</code>, <code>np.concatenate</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="np.zeros,np.where,Tensor.min,Tensor.max"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.zeros</code> · <code>np.where</code> · <code>Tensor.min</code> · <code>Tensor.max</code></summary>
<ul><li><code>np.zeros(shape, dtype=float)</code> &nbsp; A new array of the given shape filled with 0.0 (float64). The standard way to allocate an output array before filling it in a loop.</li><li><code>np.where(condition)  or  np.where(condition, x, y)</code> &nbsp; With one argument: the indices where condition is True, as a tuple with one index array per dimension (take [0] for a 1-D array). With three: a new array that takes x where condition is True and y elsewhere, element by element.</li><li><code>x.min() / x.min(dim)</code> &nbsp; x.min() is the smallest element (a 0-dimensional tensor); x.min(dim) works along one dimension and returns two tensors, the minimum values and their positions. NumPy arrays have the same method: arr.min() and arr.min(axis=...), which returns only the values.</li><li><code>x.max() / x.max(dim)</code> &nbsp; x.max() is the largest element (a 0-dimensional tensor); x.max(dim) works along one dimension and returns two tensors, the maximum values and their positions (indices). NumPy arrays have the same method: arr.max() and arr.max(axis=...), which returns only the values.</li></ul></details>

In [ ]:
from scipy import ndimage

ex = day3.load_pet_example("Bengal_105")
img, tri = ex["image"], ex["trimap"]
target = day3.trimap_to_target(tri)                    # 0 background, 1 pet, 2 border
pet = target > 0                                       # pet + its border
inst, n_inst = ndimage.label(ndimage.binary_opening(pet, iterations=2))   # connected components = instances here
print("image", img.shape, "| raw trimap values", np.unique(tri), "| instances found:", n_inst,
      "| head boxes:", len(ex["head_boxes"]))

fig, axes = plt.subplots(1, 4, figsize=(15, 4.3))
axes[0].imshow(img); axes[0].set_title(f"image ({ex['breed']})")
# semantic: one colour per CLASS
sem_rgb = np.zeros(img.shape); sem_rgb[target == 1] = day3.SEG_COLORS[1]; sem_rgb[target == 2] = [0.8, 0.8, 0.8]
axes[1].imshow(img); axes[1].imshow(np.dstack([sem_rgb, 0.65 * (target > 0)]))
axes[1].set_title("semantic: every pet pixel = 'cat'\n(grey = uncertain border)")
# instance: one colour per OBJECT, plus boxes
axes[2].imshow(img)
inst_rgb = np.zeros(img.shape); inst_alpha = np.zeros(img.shape[:2])
for k in range(1, n_inst + 1):
    inst_rgb[inst == k] = day3.SEG_COLORS[k + 1]; inst_alpha[inst == k] = 0.6
    ys, xs = np.where(inst == k)
    axes[2].add_patch(Rectangle((xs.min(), ys.min()), xs.max() - xs.min(), ys.max() - ys.min(),
                                fill=False, ec=day3.SEG_COLORS[k + 1], lw=2))
    axes[2].text(xs.min(), ys.min() - 5, f"cat #{k}", color="white", fontsize=9,
                 bbox=dict(fc=day3.SEG_COLORS[k + 1], ec="none", pad=1))
axes[2].imshow(np.dstack([inst_rgb, inst_alpha]))
viz.draw_boxes(axes[2], ex["head_boxes"], edgecolor="white", lw=1)
axes[2].set_title("instance: one mask + box per cat\n(white = head boxes from the XML)")
# panoptic: things with ids, stuff without
pan = np.zeros(img.shape); pan[...] = [0.55, 0.55, 0.45]
for k in range(1, n_inst + 1):
    pan[inst == k] = day3.SEG_COLORS[k + 1]
axes[3].imshow(pan); axes[3].set_title("panoptic: every pixel labelled\nstuff 'background' + things 'cat #1/#2'")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

> Instance segmentation is not "semantic + connected components" in general: two
> touching cats form one blob. Here they do not touch, so `ndimage.label` recovers the
> instances; D3-03 trains Mask R-CNN to separate touching pedestrians.

### How masks are stored

| format | what | used by |
|---|---|---|
| dense mask PNG | one integer per pixel | Pet trimaps, Cityscapes, ADE20K, Penn-Fudan |
| polygon | list of (x, y) vertices per object | COCO (`segmentation: [[x1, y1, ...]]`), LabelMe, CVAT |
| RLE | run lengths of 0s and 1s | COCO crowd regions and *all* model outputs (`pycocotools`), SAM |

Polygons are tiny and editable but cannot express holes easily; RLE is lossless and
compresses large blobs very well.

In [ ]:
import contourpy
small = pet[::2, ::2]                                         # half resolution to keep the plot light
lines = contourpy.contour_generator(z=small.astype(float)).lines(0.5)
poly = max(lines, key=len)                                    # outline of the largest blob
poly_s = poly[::max(len(poly) // 60, 1)]                     # an annotator would click ~60 points

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(small, cmap="gray"); axes[0].set_title(f"dense mask: {small.size:,} values")
axes[1].imshow(img[::2, ::2]); axes[1].plot(poly_s[:, 0], poly_s[:, 1], "-o", ms=3, color="#F58518")
axes[1].set_title(f"polygon: {len(poly_s)} vertices ({2 * len(poly_s)} numbers)")
axes[2].imshow(tri, cmap="gray", vmin=0, vmax=255); axes[2].set_title("raw trimap PNG shown as 0-255:\nvalues 1, 2, 3 look black!")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

### Exercise 1 · run-length encoding

COCO's *uncompressed RLE* reads the mask **column by column** (Fortran order) and stores
the lengths of alternating runs, **always starting with a run of 0s** (length 0 if the
first pixel is 1). Example: the 2x3 mask `[[0, 1, 1], [0, 1, 0]]` read column-wise is
`0 0 | 1 1 | 1 0` -> runs `0 0`, `1 1 1`, `0` -> `counts = [2, 3, 1]`.

Implement `rle_encode(mask) -> {"size": [H, W], "counts": [...]}` and `rle_decode(rle)`.

<details class="api-ref" data-api="np.shape"><summary><b>Functions used here</b> (click to open): <code>np.shape</code></summary>
<p><b><code>np.shape(a)</code></b><br>The shape of anything array-like, as a tuple: it works on NumPy arrays and on nested Python lists, which have no .shape attribute. The RLE exercise uses it so that rle_encode accepts a list as well as an array.</p><pre>&gt;&gt;&gt; np.shape([[0, 1, 1], [0, 1, 0]])             # a nested list: 2 rows, 3 columns
(2, 3)
&gt;&gt;&gt; np.shape(np.zeros((4, 5)))
(4, 5)</pre><p>See also: <code>Tensor.shape</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="np.asarray,Tensor.reshape,np.array_equal"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.asarray</code> · <code>Tensor.reshape</code> · <code>np.array_equal</code></summary>
<ul><li><code>np.asarray(obj)</code> &nbsp; Converts its argument (a list, a PIL image, a CPU tensor, ...) to a NumPy array; an array is returned as it is, without a copy. A grey PIL image becomes a (H, W) uint8 array, a colour one (H, W, 3).</li><li><code>x.reshape(*shape)</code> &nbsp; Returns the same elements in a new shape, read in row order (the last dimension changes fastest). One size may be -1: it is computed from the others. NumPy arrays have the same method.</li><li><code>np.array_equal(a, b)</code> &nbsp; True if the two arrays have the same shape and exactly the same elements, as one Python bool (a == b gives an array of element-wise answers instead). For floats with rounding errors use np.allclose.</li></ul></details>

In [ ]:
def rle_encode(mask):
    '''mask (H, W) of 0 / 1 (or bool) -> {"size": [H, W], "counts": [run lengths]}: COCO's uncompressed RLE.'''
    mask = np.asarray(mask).astype(np.uint8)            # (H, W), values 0 / 1
    H, W = mask.shape
    flat = mask.reshape(-1, order="F")                  # (H * W,) column by column, like COCO
    counts = []
    current = 0                                         # the value of the run being counted: the first run is 0s
    run = 0                                             # its length so far (stays 0 if the mask starts with a 1)
    for value in flat:
        # YOUR CODE: if value == current, the run is one pixel longer (run += 1); otherwise the run is
        #            over: append run to counts, then a new run of `value` starts (current = value, run = 1)
        raise NotImplementedError
    # YOUR CODE: the last run is still open: append it to counts
    return {"size": [int(H), int(W)], "counts": counts}

def rle_decode(rle):
    '''{"size": [H, W], "counts": [...]} -> mask (H, W) uint8 of 0 / 1: the inverse of rle_encode.'''
    H, W = rle["size"]
    flat = np.zeros(H * W, np.uint8)                    # (H * W,) all 0 to start
    pos = 0                                             # where the next run starts in flat
    value = 0                                           # the value of the next run: 0s first
    for length in rle["counts"]:
        # YOUR CODE: write value into the next `length` pixels, flat[pos:pos + length]
        # YOUR CODE: move pos to the end of this run
        # YOUR CODE: the next run has the other value: value = 1 - value
        raise NotImplementedError
    return flat.reshape((H, W), order="F")              # back to (H, W), column by column

tests.check_rle(rle_encode, rle_decode)

In [ ]:
rle = rle_encode(pet)
back = rle_decode(rle)
print(f"pet mask {pet.shape}: {pet.size:,} pixels -> {len(rle['counts'])} run lengths "
      f"({pet.size / len(rle['counts']):.0f}x smaller), lossless: {np.array_equal(back, pet)}")
print("first counts:", rle["counts"][:12], "...")

## 2 · Metrics: pixel accuracy lies

The obvious metric, the fraction of correctly labelled pixels, is dominated by the
biggest class. A model that predicts **"background everywhere"** on an image where the
object covers 2% of the pixels scores **98% pixel accuracy** and is useless.

The fix is to score **each class separately** with the intersection over union you know
from boxes, now computed on pixel sets, and average over classes:

$$\text{IoU}_c = \frac{TP_c}{TP_c + FP_c + FN_c}, \qquad \text{mIoU} = \frac{1}{C}\sum_c \text{IoU}_c,
\qquad \text{Dice}_c = \frac{2\,TP_c}{2\,TP_c + FP_c + FN_c}$$

<details class="api-ref" data-api="np.zeros_like,np.nanmean"><summary><b>Functions used here</b> (click to open): <code>np.zeros_like</code> · <code>np.nanmean</code></summary>
<p><b><code>np.zeros_like(a, dtype=None)</code></b><br>A new array of zeros with the same shape and dtype as a (dtype= overrides the type). The easy way to start an empty mask or an accumulator that matches an existing image.</p><pre>&gt;&gt;&gt; gt = np.array([[0, 1], [1, 1]])
&gt;&gt;&gt; np.zeros_like(gt)
array([[0, 0],
       [0, 0]])
&gt;&gt;&gt; np.zeros_like(gt, bool)                      # same shape, boolean
array([[False, False],
       [False, False]])</pre><p><i>Watch out:</i> It copies the dtype too: zeros_like of an integer mask is an integer array, so writing 0.5 into it stores 0.</p><p>See also: <code>np.zeros</code>, <code>torch.zeros_like</code></p><p><b><code>np.nanmean(a)</code></b><br>The mean of an array that skips NaN (&quot;not a number&quot;) values. mIoU is averaged this way: a class absent from both prediction and target has an undefined IoU (NaN) and is left out, instead of turning the whole mean into NaN.</p><pre>&gt;&gt;&gt; iou = np.array([0.9, np.nan, 0.5])           # class 1 appears nowhere
&gt;&gt;&gt; float(np.mean(iou))
nan
&gt;&gt;&gt; float(np.nanmean(iou))                       # (0.9 + 0.5) / 2
0.7</pre><p>See also: <code>np.isnan</code>, <code>np.mean</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="np.arange,Tensor.mean"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.arange</code> · <code>Tensor.mean</code></summary>
<ul><li><code>np.arange(stop) / np.arange(start, stop, step, dtype=None)</code> &nbsp; Like Python&#x27;s range, as a NumPy array: from start (default 0) up to but not including stop. With reshape, the quickest way to build a small test array with distinct values.</li><li><code>x.mean(dim=None, keepdim=False)</code> &nbsp; The average of all elements, or along dim (a number or a tuple), which then disappears from the shape. For a batch (B, C, H, W), mean(dim=(2, 3)) averages over the pixels and gives (B, C): global average pooling. NumPy arrays have the same method, with axis= for dim=.</li></ul></details>

In [ ]:
H = W = 128
gt = np.zeros((H, W), int); yy, xx = np.mgrid[:H, :W]
gt[(yy - 70) ** 2 + (xx - 50) ** 2 <= 10 ** 2] = 1            # a small object: ~2% of the image
lazy = np.zeros_like(gt)                                       # predicts background everywhere
shifted = np.zeros_like(gt); shifted[(yy - 72) ** 2 + (xx - 54) ** 2 <= 11 ** 2] = 1   # decent but offset

fig, axes = plt.subplots(1, 4, figsize=(14, 3.6), gridspec_kw=dict(width_ratios=[1, 1, 1, 1.5]))
rows = []
for ax, (name, p) in zip(axes, [("ground truth", gt), ("'all background'", lazy), ("offset circle", shifted)]):
    ax.imshow(day3.colorize(p)); ax.axis("off")
    acc = seg.pixel_accuracy(p, gt); iou = seg.iou_per_class(p, gt, 2); miou = np.nanmean(iou)
    ax.set_title(f"{name}\nacc {acc:.3f} | IoU(obj) {iou[1]:.2f} | mIoU {miou:.2f}", fontsize=9)
    rows.append((name, acc, iou[1], miou))
names = [r[0] for r in rows[1:]]
x = np.arange(2)
for k, (lab, col) in enumerate([("pixel accuracy", "#BAB0AC"), ("IoU of the object", "#E45756"), ("mIoU", "#4C78A8")]):
    axes[3].bar(x + (k - 1) * 0.27, [r[k + 1] for r in rows[1:]], 0.27, label=lab, color=col)
axes[3].set_xticks(x); axes[3].set_xticklabels(names); axes[3].set_ylim(0, 1.1); axes[3].legend(fontsize=8, loc="upper right")
axes[3].set_title("accuracy cannot tell them apart; IoU can")
plt.tight_layout(); plt.show()
print(f"object covers {gt.mean():.1%} of the image")

Compare the two predictions in the bar chart: the model that found nothing and the decent offset
circle both score about 0.98 pixel accuracy; only the IoU of the object (0 vs 0.57) tells them apart.

### Lab · segmentation metrics
**Predict first:** a perfect prediction is shifted one pixel to the right. Which loses more IoU, a large
blob or a vessel 2 pixels wide, and roughly how much? Pick the presets in the lab to check, then work through
the "Try this" list under it (4-5 minutes).

In [ ]:
labs.show("seg-metrics")

### Exercise 2 · Dice coefficient

For two binary masks $A$ (prediction) and $B$ (target): $\text{Dice} = \frac{2|A \cap B|}{|A| + |B|}$
(in the lab's terms, $2TP / (2TP + FP + FN)$). Two lines with NumPy boolean arrays; return a Python
`float`, with `eps` added to numerator and denominator so that two empty masks score 1.

<details class="api-ref" data-api="np.logical_and"><summary><b>Functions you will need</b> (click to open): <code>np.logical_and</code></summary>
<p><b><code>np.logical_and(a, b)</code></b><br>Element-wise AND of two boolean (or 0/1) arrays of the same shape: True where both are true. On a predicted and a true mask it is their intersection, and .sum() counts its pixels; the same as a &amp; b for boolean arrays.</p><pre>&gt;&gt;&gt; pred = np.array([[1, 1, 0], [0, 1, 0]], bool)
&gt;&gt;&gt; gt   = np.array([[0, 1, 1], [0, 1, 0]], bool)
&gt;&gt;&gt; np.logical_and(pred, gt)
array([[False,  True, False],
       [False,  True, False]])
&gt;&gt;&gt; int(np.logical_and(pred, gt).sum())          # intersection: 2 pixels
2</pre><p><i>Watch out:</i> It treats every non-zero value as True. a &amp; b on integer arrays is a bitwise AND instead: 2 &amp; 1 is 0, so convert masks to bool first.</p><p>See also: <code>np.logical_or</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="Tensor.sum"><summary><b>Functions you will need (met before)</b> (click to open): <code>Tensor.sum</code></summary>
<ul><li><code>x.sum(dim=None, keepdim=False)</code> &nbsp; Adds up all elements (giving a 0-dimensional tensor), or only along dim, which then disappears from the shape: for a (2, 3) tensor, sum(dim=0) adds the two rows together and gives one value per column, shape (3,). dim can be a tuple; keepdim=True keeps the summed dimension with size 1.</li></ul></details>

In [ ]:
def dice_coefficient(pred_mask, target_mask, eps=1e-7):
    '''two masks of the same shape (anything that converts to bool) -> Dice = 2|A and B| / (|A| + |B|), a float.'''
    p = np.asarray(pred_mask, bool)                     # A, the prediction
    t = np.asarray(target_mask, bool)                   # B, the target
    # 1. YOUR CODE: inter, the number of pixels that are True in both (np.logical_and, then .sum())
    # 2. YOUR CODE: return (2 * inter + eps) / (p.sum() + t.sum() + eps) as a Python float
    raise NotImplementedError

tests.check_dice(dice_coefficient)

### The confusion matrix holds every metric

For $C$ classes, count pixels in a $C \times C$ matrix: row = true class, column =
predicted class. The diagonal is $TP_c$; the rest of column $c$ is $FP_c$; the rest of
row $c$ is $FN_c$. (The 2x2 matrix in the lab is the case $C = 2$.) Below: a shapes image
and a deliberately flawed "prediction" (the square shifted, the triangle called a square,
the circles shrunk).

<details class="api-ref" data-api="scipy.ndimage.binary_erosion,np.roll,np.log1p,np.isnan"><summary><b>Functions used here</b> (click to open): <code>scipy.ndimage.binary_erosion</code> · <code>np.roll</code> · <code>np.log1p</code> · <code>np.isnan</code></summary>
<p><b><code>ndimage.binary_erosion(mask, iterations=1)</code></b><br>Shrinks the True regions of a binary mask: a pixel stays True only if it and its 4 side neighbours are all True, so each iteration peels one pixel off every border. D3-01 uses it to make a deliberately too small &quot;prediction&quot;; mask &amp; ~binary_erosion(mask) is the mask&#x27;s outline.</p><pre>&gt;&gt;&gt; from scipy import ndimage
&gt;&gt;&gt; m = np.zeros((5, 5), bool); m[1:4, 1:4] = True       # a 3x3 square
&gt;&gt;&gt; ndimage.binary_erosion(m).astype(int)                # only the centre survives
array([[0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0]])
&gt;&gt;&gt; int((m &amp; ~ndimage.binary_erosion(m)).sum())          # the outline: 8 pixels
8</pre><p><i>Watch out:</i> Pixels outside the array count as False, so a mask that touches the image border also shrinks from there.</p><p>See also: <code>scipy.ndimage.binary_opening</code></p><p><b><code>np.roll(a, shift, axis=None)</code></b><br>Shifts the elements of an array by shift positions along an axis; what falls off one end comes back at the other (it wraps around). With tuples, np.roll(mask, (4, 6), (0, 1)) moves a mask 4 pixels down and 6 pixels to the right.</p><pre>&gt;&gt;&gt; np.roll(np.array([1, 2, 3, 4, 5]), 2)
array([4, 5, 1, 2, 3])
&gt;&gt;&gt; m = np.array([[1, 0, 0],
...               [0, 0, 0]])
&gt;&gt;&gt; np.roll(m, (1, 1), axis=(0, 1))              # one row down, one column right
array([[0, 0, 0],
       [0, 1, 0]])</pre><p><i>Watch out:</i> Without axis the array is flattened, shifted and reshaped back, which mixes rows.</p><p><b><code>np.log1p(x)</code></b><br>log(1 + x), element-wise (natural logarithm), accurate even for tiny x. Handy to display counts that range from 0 to thousands, such as a confusion matrix: log1p(0) is 0, so empty cells stay 0 instead of becoming -inf.</p><pre>&gt;&gt;&gt; cm = np.array([[0, 9], [99, 9999]])
&gt;&gt;&gt; np.log1p(cm).round(2)                        # log(1), log(10), log(100), log(10000)
array([[0.  , 2.3 ],
       [4.61, 9.21]])</pre><p>See also: <code>np.log10</code>, <code>torch.log</code></p><p><b><code>np.isnan(a)</code></b><br>True where a value is NaN (&quot;not a number&quot;, e.g. the result of 0 / 0), element-wise. You need it because NaN is not equal to anything, not even to NaN, so a comparison with == never finds one. In D3-01 the IoU of a class that appears nowhere is NaN.</p><pre>&gt;&gt;&gt; iou = np.array([0.9, np.nan, 0.5])
&gt;&gt;&gt; np.isnan(iou)
array([False,  True, False])
&gt;&gt;&gt; iou == np.nan                                # never works
array([False, False, False])</pre><p>See also: <code>np.nanmean</code>, <code>np.nan_to_num</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="np.random.default_rng,np.ndenumerate,np.nan_to_num"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.random.default_rng</code> · <code>np.ndenumerate</code> · <code>np.nan_to_num</code></summary>
<ul><li><code>rng = np.random.default_rng(seed)</code> &nbsp; Creates a NumPy random number generator; the same seed gives the same numbers on every run. Draw from it with rng.normal(size=...) (normal distribution), rng.random(size) (uniform in [0, 1)), rng.integers(low, high, size) or rng.permutation(n).</li><li><code>for index, value in np.ndenumerate(a):</code> &nbsp; Loops over every element of an array of any shape, giving its index tuple together with its value (enumerate for N-D arrays). D2-03 uses it to write each grid cell&#x27;s value on a plot.</li><li><code>np.nan_to_num(a, nan=0.0)</code> &nbsp; Returns a copy of a with every NaN replaced by 0 (or by the value given as nan=), and infinities by very large finite numbers. Used before code that cannot handle NaN, such as writing the values into a figure.</li></ul></details>

In [ ]:
rng = np.random.default_rng(7)
s_img, s_boxes, s_labels, s_mask = shapes.make_sample(rng, 128, max_objects=4)
pred = s_mask.copy()
objs = [(k, b, l) for k, (b, l) in enumerate(zip(s_boxes, s_labels))]
for k, b, l in objs:
    region = np.zeros_like(s_mask, bool); x1, y1, x2, y2 = b.astype(int); region[y1:y2, x1:x2] = s_mask[y1:y2, x1:x2] == l + 1
    if l == 2:   pred[region] = 2                                            # triangle -> square
    elif l == 0: pred[region & ~ndimage.binary_erosion(region, iterations=3)] = 0   # circle shrunk
    else:        pred[region] = 0; pred[np.roll(region, (4, 6), (0, 1))] = l + 1   # square shifted
cm = seg.confusion_matrix(pred, s_mask, 4)
names4 = ["bg"] + shapes.CLASS_NAMES

fig, axes = plt.subplots(1, 4, figsize=(15, 3.8), gridspec_kw=dict(width_ratios=[1, 1, 1.2, 1.2]))
axes[0].imshow(s_img); axes[0].set_title("image"); axes[1].imshow(day3.colorize(pred)); axes[1].set_title("flawed prediction")
for ax in axes[:2]: ax.axis("off")
im = axes[2].imshow(np.log1p(cm), cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    axes[2].text(j, i, f"{v}", ha="center", va="center", fontsize=8, color="white" if np.log1p(v) > np.log1p(cm).max() * 0.6 else "black")
axes[2].set_xticks(range(4)); axes[2].set_xticklabels(names4, rotation=30); axes[2].set_yticks(range(4)); axes[2].set_yticklabels(names4)
axes[2].set_xlabel("predicted"); axes[2].set_ylabel("true"); axes[2].set_title("confusion matrix (pixels)")
iou = seg.iou_per_class(pred, s_mask, 4)
axes[3].bar(names4, np.nan_to_num(iou), color=[day3.SEG_COLORS[i] for i in range(4)])
for i, v in enumerate(iou):
    axes[3].text(i, np.nan_to_num(v) + 0.02, "absent" if np.isnan(v) else f"{v:.2f}", ha="center", fontsize=9)
axes[3].set_ylim(0, 1.12); axes[3].set_title(f"per-class IoU -> mIoU = {np.nanmean(iou):.3f}")
plt.tight_layout(); plt.show()
print(f"pixel accuracy {seg.pixel_accuracy(pred, s_mask):.3f} vs mIoU {np.nanmean(iou):.3f}")

Read the off-diagonal cells: every triangle pixel sits in row "triangle", column "square", so the
triangle's IoU is 0, while pixel accuracy barely notices because the background dominates the count.

### Exercise 3 · mIoU from the confusion matrix

Use the matrix (rows = target, columns = prediction). A class absent from both the
prediction and the target has an undefined IoU (`nan`) and is skipped by `np.nanmean`.

<details class="api-ref" data-api="np.diag,np.nanmean"><summary><b>Functions you will need (met before)</b> (click to open): <code>np.diag</code> · <code>np.nanmean</code></summary>
<ul><li><code>np.diag(a)</code> &nbsp; With a 2-D array: returns its diagonal a[0, 0], a[1, 1], ... as a 1-D array. With a 1-D array: builds a square matrix with those values on the diagonal. D2-01 takes the diagonal of an (N, N) IoU matrix to get the IoU of each aligned pair (prediction i with ground truth i).</li><li><code>np.nanmean(a)</code> &nbsp; The mean of an array that skips NaN (&quot;not a number&quot;) values. mIoU is averaged this way: a class absent from both prediction and target has an undefined IoU (NaN) and is left out, instead of turning the whole mean into NaN.</li></ul></details>

In [ ]:
def mean_iou(pred, target, num_classes):
    '''pred, target: integer masks of the same shape, values 0 .. num_classes-1 -> mIoU (a float),
    the mean IoU over the classes present in the prediction or the target.'''
    cm = seg.confusion_matrix(pred, target, num_classes).astype(float)   # (C, C) rows = target, cols = prediction
    # 1. YOUR CODE: tp, the true positives of every class: the diagonal of cm (np.diag) -> (C,)
    # 2. YOUR CODE: fp, pixels predicted as class c that are something else: column sums (cm.sum(0)) - tp -> (C,)
    # 3. YOUR CODE: fn, pixels of class c predicted as something else: row sums (cm.sum(1)) - tp -> (C,)
    # 4. YOUR CODE: iou, an empty list; for every class c: union = tp[c] + fp[c] + fn[c]; append
    #               tp[c] / union if union > 0, otherwise np.nan (class absent from both: undefined)
    # 5. YOUR CODE: return the mean that skips the nan values, float(np.nanmean(iou))
    raise NotImplementedError

tests.check_mean_iou(mean_iou)

### Dice vs IoU

Both count the same pixels, so they are monotonically related: with $I$ = IoU,
$\text{Dice} = \frac{2I}{1 + I}$. Dice is always $\geq$ IoU (it looks more generous), and
they agree at 0 and 1 (the curve in the lab: every mask you painted sat on it). Rankings of models are identical *per image*; averaged over a
dataset they can differ slightly. Medical imaging reports Dice, natural-image benchmarks
report mIoU - convert before you compare papers.

<details class="api-ref" data-api="np.array,np.linspace,np.clip"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.array</code> · <code>np.linspace</code> · <code>np.clip</code></summary>
<ul><li><code>np.array(data, dtype=None)</code> &nbsp; Creates a NumPy array from a (nested) list, copying the data. The dtype is inferred: integers give int64, any float gives float64; dtype=float gives floats from integer literals.</li><li><code>np.linspace(start, stop, num=50)</code> &nbsp; num evenly spaced numbers from start to stop, both ends included. Used for plot axes and grids.</li><li><code>np.clip(a, a_min, a_max)</code> &nbsp; Limits every element to [a_min, a_max]: smaller values become a_min, larger ones a_max. Keeps simulated scores inside [0, 1], and probabilities away from 0 before a log.</li></ul></details>

In [ ]:
rng = np.random.default_rng(0)
pts = []
for _ in range(300):
    a = np.zeros((64, 64), bool); b = np.zeros((64, 64), bool)
    x, y, w, h = rng.integers(0, 40, 4); a[y:y + 10 + h // 2, x:x + 10 + w // 2] = True
    x, y, w, h = np.clip([x, y, w, h] + rng.integers(-12, 12, 4), 0, 40); b[y:y + 10 + h // 2, x:x + 10 + w // 2] = True
    pts.append((seg.iou_per_class(a.astype(int), b.astype(int), 2)[1], seg.dice_coefficient(a, b)))
pts = np.array(pts)
i = np.linspace(0, 1, 200)
fig, ax = plt.subplots(figsize=(5.5, 4.2))
ax.plot(i, 2 * i / (1 + i), lw=2.5, label="Dice = 2 IoU / (1 + IoU)")
ax.plot(i, i, "--", color="grey", label="Dice = IoU")
ax.scatter(pts[:, 0], pts[:, 1], s=8, color="#F58518", alpha=0.6, label="300 random mask pairs")
ax.annotate("IoU 0.5 is Dice 0.67", (0.5, 2 / 3), xytext=(0.55, 0.35), arrowprops=dict(arrowstyle="->"), fontsize=9)
ax.set_xlabel("IoU"); ax.set_ylabel("Dice"); ax.legend(fontsize=8, loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3 · Losses

A segmentation network outputs a logit **per pixel and class**: shape `(N, C, H, W)`.
The default loss is the classification loss applied at every pixel:

* **cross-entropy** (`F.cross_entropy(logits, target)` with `target (N, H, W)` integers) for
  multi-class; **BCE with logits** for a single foreground channel;
* optional **class weights** (`weight=`) to up-weight rare classes;
* **soft Dice loss**: replace the hard sets in Dice by probabilities so it is differentiable,
  $\;\mathcal{L}_\text{Dice} = 1 - \frac{2\sum_i p_i t_i + \epsilon}{\sum_i p_i + \sum_i t_i + \epsilon}$;
* **focal loss** (Day 2): down-weights easy pixels by $(1 - p_t)^\gamma$;
* in practice **BCE + Dice** (or CE + Dice): CE gives smooth per-pixel gradients, Dice
  directly optimises the overlap and does not care how many background pixels there are.

### Exercise 4 · soft Dice loss in PyTorch

`prob` and `target` have the same shape (any shape); compute **one** Dice over the whole
batch and return the loss $1 - \text{Dice}$ as a 0-dim tensor. Use torch operations only, so
autograd can differentiate it.

<details class="api-ref" data-api="Tensor.float,Tensor.sum"><summary><b>Functions you will need (met before)</b> (click to open): <code>Tensor.float</code> · <code>Tensor.sum</code></summary>
<ul><li><code>x.float()</code> &nbsp; Returns the tensor converted to float32, the dtype layers and losses work in (the same as x.to(torch.float32)); x itself is not changed. A uint8 image becomes 0.0 to 255.0, so divide by 255 to get [0, 1].</li><li><code>x.sum(dim=None, keepdim=False)</code> &nbsp; Adds up all elements (giving a 0-dimensional tensor), or only along dim, which then disappears from the shape: for a (2, 3) tensor, sum(dim=0) adds the two rows together and gives one value per column, shape (3,). dim can be a tuple; keepdim=True keeps the summed dimension with size 1.</li></ul></details>

In [ ]:
def soft_dice_loss(prob, target, eps=1e-6):
    '''prob: probabilities in [0, 1], target: 0 / 1, same shape (any) -> 1 - Dice over the whole batch, a 0-d tensor.'''
    target = target.float()
    # 1. YOUR CODE: inter, the soft intersection: the sum of prob * target over all elements
    # 2. YOUR CODE: total, the soft sizes: prob.sum() + target.sum()
    # 3. YOUR CODE: dice = (2 * inter + eps) / (total + eps)
    # 4. YOUR CODE: return 1 - dice
    raise NotImplementedError

tests.check_soft_dice_torch(soft_dice_loss)

### Who gets the gradient on a small object?

A 1-D "image" of 4 096 pixels holds an object of `n` pixels. The model currently predicts
$p = 0.3$ on the object (too low) and $0.1$ on the background (slightly too high). We let
autograd compute $\partial \mathcal{L} / \partial p_i$ for every pixel and ask: **what share of
the total gradient comes from the object pixels?** With mean BCE, each pixel has the
same voice, so 4 000 background pixels outvote a 40-pixel object. Dice normalises by the
object size.

<details class="api-ref" data-api="torch.where,F.binary_cross_entropy,torch.log,np.logspace,np.log10"><summary><b>Functions used here</b> (click to open): <code>torch.where</code> · <code>F.binary_cross_entropy</code> · <code>torch.log</code> · <code>np.logspace</code> · <code>np.log10</code></summary>
<p><b><code>torch.where(cond, a, b)  /  torch.where(cond)</code></b><br>With three arguments: an element-wise choice, a where cond is True and b where it is False (a and b broadcast, so they can be scalars or single-value tensors). With only cond: the indices of the True elements, one tensor per dimension (rows, then columns), like np.nonzero.</p><pre>&gt;&gt;&gt; t = torch.tensor([1., 0., 0., 1.])                         # 1 = object pixel
&gt;&gt;&gt; torch.where(t &gt; 0, torch.tensor(0.3), torch.tensor(0.1))   # 0.3 on the object, 0.1 elsewhere
tensor([0.3000, 0.1000, 0.1000, 0.3000])
&gt;&gt;&gt; ys, xs = torch.where(torch.tensor([[0, 1], [1, 1]]) &gt; 0)
&gt;&gt;&gt; ys, xs
(tensor([0, 1, 1]), tensor([1, 0, 1]))</pre><p><i>Watch out:</i> Gradients flow only into the chosen elements, but a NaN produced in the branch that was not chosen (log(0), 0 / 0) still turns the gradient into NaN: make both branches safe.</p><p>See also: <code>np.where</code>, <code>np.nonzero</code></p><p><b><code>F.binary_cross_entropy(prob, target)</code></b><br>Binary cross-entropy for probabilities: prob (values in (0, 1)) and target (0s and 1s, as floats) have the same shape; the result is the mean of -[t log p + (1 - t) log(1 - p)] over all elements. D3-01 uses it to ask how big the gradient is with respect to each probability. To train a model, use F.binary_cross_entropy_with_logits on the raw outputs instead.</p><pre>&gt;&gt;&gt; p = torch.tensor([0.9, 0.2, 0.5])
&gt;&gt;&gt; t = torch.tensor([1., 0., 1.])
&gt;&gt;&gt; F.binary_cross_entropy(p, t)                 # mean of -log 0.9, -log 0.8, -log 0.5
tensor(0.3406)</pre><p><i>Watch out:</i> It wants probabilities, not logits (values outside [0, 1] raise an error), and a float target. A sigmoid followed by this loss is less stable than F.binary_cross_entropy_with_logits.</p><p>See also: <code>F.binary_cross_entropy_with_logits</code>, <code>torch.sigmoid</code></p><p><b><code>torch.log(x)</code></b><br>Natural logarithm, element-wise. log(0) is -inf and the log of a negative number is nan, so a loss written by hand (the focal loss in D3-01) must keep its probabilities strictly above 0.</p><pre>&gt;&gt;&gt; torch.log(torch.tensor([1.0, 0.5, 0.0]))
tensor([ 0.0000, -0.6931,    -inf])</pre><p><i>Watch out:</i> -torch.log(torch.sigmoid(z)) is unstable for very negative z; the built-in losses (F.binary_cross_entropy_with_logits, F.cross_entropy) compute the same quantity safely.</p><p>See also: <code>np.log1p</code>, <code>F.binary_cross_entropy</code></p><p><b><code>np.logspace(start, stop, num)</code></b><br>num numbers from 10**start to 10**stop, evenly spaced on a log scale: each one is the previous times a constant factor. Use it to sweep a quantity over several orders of magnitude, such as object sizes from 4 to 2048 pixels.</p><pre>&gt;&gt;&gt; np.logspace(0, 3, 4)                         # 10**0 ... 10**3
array([   1.,   10.,  100., 1000.])
&gt;&gt;&gt; np.logspace(np.log10(4), np.log10(64), 5)    # from 4 to 64: times 2 at each step
array([ 4.,  8., 16., 32., 64.])</pre><p><i>Watch out:</i> start and stop are exponents, not values: np.logspace(4, 2048, 30) means 10**4 to 10**2048. np.geomspace(4, 2048, 30) takes the end values directly.</p><p>See also: <code>np.linspace</code>, <code>np.log10</code></p><p><b><code>np.log10(x)</code></b><br>Base-10 logarithm, element-wise: log10(1000) is 3. D3-01 uses it to turn sizes into the exponents np.logspace expects.</p><pre>&gt;&gt;&gt; np.log10(np.array([1., 10., 1000.]))
array([0., 1., 3.])</pre><p>See also: <code>np.logspace</code>, <code>np.log1p</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="torch.zeros,torch.tensor,Tensor.requires_grad_,Tensor.backward,Tensor.grad,Tensor.abs"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torch.zeros</code> · <code>torch.tensor</code> · <code>Tensor.requires_grad_</code> · <code>Tensor.backward</code> · <code>Tensor.grad</code> · <code>Tensor.abs</code></summary>
<ul><li><code>torch.zeros(*size, dtype=None, device=None)</code> &nbsp; A new tensor of the given shape filled with 0 (float32 unless dtype says otherwise). Used to allocate outputs and to build test inputs of a given shape.</li><li><code>torch.tensor(data, dtype=None, device=None)</code> &nbsp; Builds a new tensor from a Python number, a (nested) list or a NumPy array, always copying the data. The dtype is inferred: integers give int64, floats give float32; pass dtype= to choose.</li><li><code>p.requires_grad_(flag=True)</code> &nbsp; Sets, in place, whether autograd computes a gradient for this tensor, and returns the tensor; p.requires_grad_(False) does the same as p.requires_grad = False. Freezing a layer means this for all its parameters: backward() then computes no .grad for them, so the optimizer leaves them unchanged.</li><li><code>loss.backward()</code> &nbsp; Computes the gradient of a single number (usually the loss) with respect to every tensor it was computed from that has requires_grad=True, and adds it to that tensor&#x27;s .grad. One call fills the gradients of all the parameters of a model.</li><li><code>x.grad</code> &nbsp; An attribute: after backward(), the gradient of the loss with respect to x, a tensor of x&#x27;s shape. It is None before the first backward() and after optimizer.zero_grad(); every further backward() adds to it.</li><li><code>x.abs()</code> &nbsp; The absolute value of every element. (a - b).abs().max() is the largest difference between two tensors: the usual check that two results agree.</li></ul></details>

In [ ]:
def grad_share(loss_name, n_obj, N=4096, p_obj=0.3, p_bg=0.1):
    t = torch.zeros(N); t[:n_obj] = 1
    p = torch.where(t > 0, torch.tensor(float(p_obj)), torch.tensor(float(p_bg))).requires_grad_(True)
    if loss_name == "BCE":   loss = F.binary_cross_entropy(p, t)
    elif loss_name == "Dice": loss = soft_dice_loss(p, t)
    else:
        p_t = torch.where(t > 0, p, 1 - p); a_t = torch.where(t > 0, torch.tensor(0.25), torch.tensor(0.75))
        loss = (-a_t * (1 - p_t) ** 2 * torch.log(p_t)).mean()
    loss.backward()
    g = p.grad.abs()
    return float(g[:n_obj].sum() / g.sum()), float(g[0])

sizes = np.unique(np.logspace(np.log10(4), np.log10(2048), 30).astype(int))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, col in [("BCE", "#4C78A8"), ("focal (gamma 2)", "#B279A2"), ("Dice", "#E45756")]:
    key = "focal" if name.startswith("focal") else name
    share = [grad_share(key, n)[0] for n in sizes]
    axes[0].plot(100 * sizes / 4096, share, lw=2.2, label=name, color=col)
    pp = np.linspace(0.02, 0.98, 60)
    g1 = [grad_share(key, 41, p_obj=v)[1] for v in pp]
    axes[1].plot(pp, g1, lw=2.2, label=name, color=col)
axes[0].axhline(0.5, ls=":", color="grey")
axes[0].set_xscale("log"); axes[0].set_xlabel("object size (% of pixels, log)"); axes[0].set_ylabel("share of |gradient| on object pixels")
axes[0].set_title("small objects are drowned out by BCE"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_yscale("log"); axes[1].set_xlabel("predicted probability on the object pixels"); axes[1].set_ylabel("|dL/dp| for ONE object pixel")
axes[1].set_title("gradient on one pixel of a 1% object"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

Reading the plot: with BCE a 1% object receives only a few percent of the gradient.
Dice hands it about a quarter **whatever its size** (its gradient is normalised by the
object area), and focal loss helps too by silencing the easy background pixels - but
its share still collapses for the tiniest objects. The right panel shows the flip side of
focal: once a pixel is right (p -> 1) it stops learning from it. That is why Dice / BCE + Dice is the default for medical masks and
thin structures. Dice has its own failure mode: on an *empty* target its gradient is
unstable, which the BCE term smooths out.

## 4 · Getting the resolution back

We can now score a per-pixel prediction and train it with a per-pixel loss; the missing piece is a
network whose output has the resolution of the input.
A classification backbone shrinks a 224 px image to a 7x7 grid (stride 32): perfect for
"what", fatal for "where". The first fully convolutional segmenter (FCN, 2015) simply
upsampled that 7x7 map 32x. See what that does to a mask:

<details class="api-ref" data-api="F.adaptive_avg_pool2d"><summary><b>Functions used here</b> (click to open): <code>F.adaptive_avg_pool2d</code></summary>
<p><b><code>F.adaptive_avg_pool2d(x, output_size)</code></b><br>Average-pools a batch (N, C, H, W) down to the output size (h, w) you ask for; the window size and stride are worked out for you. D3-01 shrinks a mask by a factor s with it, to imitate a backbone of stride s. With output_size=1 it is global average pooling: one number per channel.</p><pre>&gt;&gt;&gt; x = torch.arange(16.).reshape(1, 1, 4, 4)
&gt;&gt;&gt; F.adaptive_avg_pool2d(x, (2, 2))             # the mean of each 2x2 block
tensor([[[[ 2.5000,  4.5000],
          [10.5000, 12.5000]]]])
&gt;&gt;&gt; F.adaptive_avg_pool2d(torch.zeros(8, 512, 7, 7), 1).shape
torch.Size([8, 512, 1, 1])</pre><p>See also: <code>nn.AdaptiveAvgPool2d</code>, <code>F.interpolate</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="F.interpolate,Tensor.numpy"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>F.interpolate</code> · <code>Tensor.numpy</code></summary>
<ul><li><code>F.interpolate(x, size=None, scale_factor=None, mode=&quot;nearest&quot;, align_corners=None)</code> &nbsp; Resizes a batch of maps (N, C, H, W) to size=(h, w), or by scale_factor. mode=&quot;nearest&quot; repeats pixels, mode=&quot;bilinear&quot; blends neighbouring pixels for a smooth result (with align_corners=False, the usual convention for images). E.g. to upsample a coarse 5 x 5 or 7 x 7 Grad-CAM map to the image size.</li><li><code>x.numpy()</code> &nbsp; Returns the tensor&#x27;s data as a NumPy array without copying: both share the same memory, so changing one changes the other. Used to hand results to Matplotlib or to NumPy code.</li></ul></details>

In [ ]:
t = torch.tensor(pet[None, None].astype(np.float32))
fig, axes = plt.subplots(1, 4, figsize=(13, 4))
axes[0].imshow(pet, cmap="gray"); axes[0].set_title(f"mask {pet.shape[0]}x{pet.shape[1]}")
for ax, s in zip(axes[1:], [8, 16, 32]):
    small_t = F.adaptive_avg_pool2d(t, (pet.shape[0] // s, pet.shape[1] // s))
    up = F.interpolate(small_t, size=pet.shape, mode="bilinear", align_corners=False)[0, 0] > 0.5
    iou = (up.numpy() & pet).sum() / (up.numpy() | pet).sum()
    ax.imshow(up, cmap="gray"); ax.set_title(f"stride {s}: {small_t.shape[-2]}x{small_t.shape[-1]} -> up\nIoU {iou:.3f}")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

Watch the IoU in the titles and the thin parts (ears, paws, tail): at stride 32 the coarse grid still
knows *what* is there, but not *where* its outline runs.

Three ways to learn an upsampling, plus one way to avoid downsampling:

| tool | idea | parameters | watch out |
|---|---|---|---|
| **transposed conv** | each input pixel *paints* a scaled copy of the kernel | yes | checkerboard when kernel % stride != 0 |
| **bilinear (or nearest) + conv** | fixed interpolation, then a normal 3x3 conv | yes (the conv) | slightly blurrier, no artefacts |
| **pixel shuffle** | predict $r^2$ channels at low res, rearrange into an $r \times$ bigger map | yes (the conv before) | needs careful init (ICNR) |
| **dilated conv** | keep the resolution, spread the kernel taps | yes | gridding if all rates equal |

### Transposed convolution = scatter-add

A normal strided convolution *gathers* a window into one output pixel. A transposed
convolution does the reverse: every **input** pixel *scatters* `x[i, j] * kernel` into
the output at position `(i * stride, j * stride)`, and overlapping contributions add up.
Output size: $(n - 1)\,s + k - 2p$.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

xs = np.array([[1., 2., 0.], [0., 1., 3.], [2., 0., 1.]])
ks = np.array([[1., 1., 1.], [1., 2., 1.], [1., 1., 1.]])
stride = 2
full = convops.transposed_conv2d(xs, ks, stride)
fig, (a_in, a_k, a_out) = plt.subplots(1, 3, figsize=(10, 3.4), gridspec_kw=dict(width_ratios=[3, 3, 7]))
a_in.imshow(xs, cmap="Blues", vmin=0, vmax=3); viz.annotate_grid(a_in, xs); a_in.set_title("input 3x3")
a_k.imshow(ks, cmap="Oranges", vmin=0, vmax=2); viz.annotate_grid(a_k, ks); a_k.set_title("kernel 3x3")
canvas = np.zeros_like(full)
im = a_out.imshow(canvas, cmap="Greens", vmin=0, vmax=full.max())
txt = [[a_out.text(j, i, "", ha="center", va="center", fontsize=8) for j in range(full.shape[1])] for i in range(full.shape[0])]
r_in = Rectangle((-0.5, -0.5), 1, 1, fill=False, ec="#E45756", lw=2.5); a_in.add_patch(r_in)
r_out = Rectangle((-0.5, -0.5), 3, 3, fill=False, ec="#E45756", lw=2.5); a_out.add_patch(r_out)
for ax in (a_in, a_k, a_out): ax.set_xticks([]); ax.set_yticks([])

def update(f):
    i, j = divmod(f, 3)
    canvas[i * stride:i * stride + 3, j * stride:j * stride + 3] += xs[i, j] * ks
    im.set_data(canvas)
    for a in range(full.shape[0]):
        for b in range(full.shape[1]):
            txt[a][b].set_text(f"{canvas[a, b]:.0f}" if canvas[a, b] else "")
    r_in.set_xy((j - 0.5, i - 0.5)); r_out.set_xy((j * stride - 0.5, i * stride - 0.5))
    a_out.set_title(f"output 7x7: add x[{i},{j}]={xs[i, j]:.0f} * kernel at ({i * stride},{j * stride})")
    return [im, r_in, r_out]

anim = FuncAnimation(fig, update, frames=9, interval=900, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

### Lab · transposed convolution
**Predict first:** stride 2, a 3x3 kernel of ones and an input of ones. Is the output constant away
from the border? If not, which values appear? Step through the stamps in the lab, hover the output
pixels, then work through the "Try this" list (4-5 minutes).

In [ ]:
labs.show("transposed-conv")

### Exercise 5 · `transposed_conv2d`

Single channel, no padding: allocate the `(H-1)*s + kh` by `(W-1)*s + kw` output and
add `x[i, j] * k` into the window starting at `(i*s, j*s)` - the stamp you stepped through in
the lab. Return the output as a float NumPy array.

In [ ]:
def transposed_conv2d(x, k, stride=1):
    '''x (H, W), kernel k (kh, kw) -> out ((H-1)*stride + kh, (W-1)*stride + kw), float:
    every input pixel adds x[i, j] * k into the output window that starts at (i*stride, j*stride).'''
    H, W = x.shape
    kh, kw = k.shape
    # 1. YOUR CODE: out, an array of zeros of the output size
    # 2. for every input pixel (row i, column j):
    for i in range(H):
        for j in range(W):
            # YOUR CODE: top = i * stride and left = j * stride, where its stamp starts in out
            # YOUR CODE: add x[i, j] * k into out[top:top + kh, left:left + kw]
            raise NotImplementedError
    return out

tests.check_transposed_conv(transposed_conv2d)

Your loop is exactly what `F.conv_transpose2d` computes. It is also a *normal*
convolution of a **zero-stuffed** input (insert `s - 1` zeros between pixels, pad by
`k - 1`) with the **flipped** kernel - the reason it was nicknamed "fractionally strided
convolution" (and, wrongly, "deconvolution").

<details class="api-ref" data-api="F.conv_transpose2d,np.pad"><summary><b>Functions used here</b> (click to open): <code>F.conv_transpose2d</code> · <code>np.pad</code></summary>
<p><b><code>F.conv_transpose2d(x, weight, stride=1, padding=0, groups=1)</code></b><br>Transposed convolution as a function: every input pixel adds a copy of the kernel, multiplied by the pixel&#x27;s value, into the output at position (i * stride, j * stride); where copies overlap they add up. x is (N, C_in, H, W) and weight is (C_in, C_out / groups, kH, kW); the output side is (H - 1) * stride + kH - 2 * padding.</p><pre>&gt;&gt;&gt; x = torch.tensor([[[[1., 2.],
...                     [3., 4.]]]])             # (1, 1, 2, 2)
&gt;&gt;&gt; k = torch.ones(1, 1, 3, 3)                   # all-ones 3x3 kernel
&gt;&gt;&gt; F.conv_transpose2d(x, k, stride=2)           # (2 - 1) * 2 + 3 = 5: four stamps, overlaps add
tensor([[[[ 1.,  1.,  3.,  2.,  2.],
          [ 1.,  1.,  3.,  2.,  2.],
          [ 4.,  4., 10.,  6.,  6.],
          [ 3.,  3.,  7.,  4.,  4.],
          [ 3.,  3.,  7.,  4.,  4.]]]])</pre><p><i>Watch out:</i> The weight is (in_channels, out_channels, kH, kW), the reverse of F.conv2d. Despite the old name &quot;deconvolution&quot; it does not undo a convolution.</p><p>See also: <code>nn.ConvTranspose2d</code>, <code>nn.Conv2d</code></p><p><b><code>np.pad(a, pad_width, mode=&quot;constant&quot;)</code></b><br>Adds a border around an array. pad_width=2 adds 2 elements before and after every axis; a pair per axis, ((top, bottom), (left, right)), sets them separately. The default fills the border with zeros: the zero padding a convolution applies at the image edges.</p><pre>&gt;&gt;&gt; np.pad(np.array([[1, 2], [3, 4]]), 1)
array([[0, 0, 0, 0],
       [0, 1, 2, 0],
       [0, 3, 4, 0],
       [0, 0, 0, 0]])
&gt;&gt;&gt; np.pad(np.ones((2, 2)), ((0, 0), (1, 2))).shape     # rows untouched; 1 column left, 2 right
(2, 5)</pre><p><i>Watch out:</i> mode=&quot;edge&quot; repeats the border pixels and mode=&quot;reflect&quot; mirrors the image instead of adding zeros.</p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="np.allclose"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.allclose</code></summary>
<ul><li><code>np.allclose(a, b, rtol=1e-05, atol=1e-08)</code> &nbsp; True if every element of a is close to the matching one of b (|a - b| &lt;= atol + rtol * |b|), as one bool. The right way to compare floats, which rarely match exactly after a computation and its inverse.</li></ul></details>

In [ ]:
out_torch = F.conv_transpose2d(torch.tensor(xs)[None, None], torch.tensor(ks)[None, None], stride=2)[0, 0].numpy()
stuffed = np.zeros((5, 5)); stuffed[::2, ::2] = xs                       # zero-stuffing
out_conv = convops.cross_correlate2d(stuffed, ks[::-1, ::-1], pad=2)      # normal conv, flipped kernel
print("loop == F.conv_transpose2d:", np.allclose(transposed_conv2d(xs, ks, 2), out_torch),
      "| == conv on zero-stuffed input:", np.allclose(out_conv, out_torch))
fig, axes = plt.subplots(1, 3, figsize=(10, 3.3))
for ax, a, t in zip(axes, [xs, np.pad(stuffed, 2), out_conv], ["input 3x3", "zero-stuffed + padded 9x9", "3x3 conv of that = 7x7 output"]):
    ax.imshow(a, cmap="Greens"); viz.annotate_grid(ax, a, fontsize=7); ax.set_title(t); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

### The checkerboard artefact

With stride 2 and kernel 3, output pixels receive contributions from **1, 2 or 4**
input pixels depending on their parity; with kernel 4 every interior pixel receives
exactly 4. Uneven overlap = a periodic pattern baked into every upsampled map, and stacking
layers makes it worse (Odena et al., Distill 2016). Count the overlaps with an all-ones
input and kernel (the "stamps per output pixel" map of the lab):

<details class="api-ref" data-api="np.outer,Tensor.expand"><summary><b>Functions used here</b> (click to open): <code>np.outer</code> · <code>Tensor.expand</code></summary>
<p><b><code>np.outer(a, b)</code></b><br>Outer product of two 1-D vectors: the matrix M[i, j] = a[i] * b[j], of shape (len(a), len(b)). np.outer(k1, k1) turns a 1-D smoothing kernel into the matching 2-D kernel.</p><pre>&gt;&gt;&gt; np.outer([1, 2, 1], [1, 2, 1])               # 1-D blur [1 2 1] -&gt; 3x3 blur kernel
array([[1, 2, 1],
       [2, 4, 2],
       [1, 2, 1]])</pre><p><b><code>x.expand(*sizes)</code></b><br>Repeats a tensor along dimensions of size 1 (or new leading dimensions) up to the given sizes, without copying: every copy reads the same numbers in memory; -1 keeps a size as it is. It is broadcasting made explicit. D3-01 gives one (k, k) kernel to 3 channels with kk.expand(3, 1, k, k).</p><pre>&gt;&gt;&gt; k = torch.tensor([[1., 2.], [3., 4.]])
&gt;&gt;&gt; k.expand(3, 1, 2, 2).shape                   # the same 2x2 kernel for 3 channels
torch.Size([3, 1, 2, 2])
&gt;&gt;&gt; torch.tensor([[1], [2]]).expand(-1, 3)       # (2, 1) -&gt; (2, 3)
tensor([[1, 1, 1],
        [2, 2, 2]])</pre><p><i>Watch out:</i> Only size-1 dimensions can be expanded (repeat tiles real data). The copies share memory, so never write into an expanded tensor; call .contiguous() or .clone() first to get real copies.</p><p>See also: <code>Tensor.contiguous</code>, <code>Tensor.unsqueeze</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="np.ones,Tensor.transpose,Tensor.contiguous,Tensor.item,Tensor.permute"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.ones</code> · <code>Tensor.transpose</code> · <code>Tensor.contiguous</code> · <code>Tensor.item</code> · <code>Tensor.permute</code></summary>
<ul><li><code>np.ones(shape)</code> &nbsp; An array of the given shape (a number or a tuple) filled with 1.0, e.g. a constant learning-rate schedule: learning rate / peak = 1 at every step.</li><li><code>x.transpose(dim0, dim1)</code> &nbsp; Swaps two dimensions of a tensor without copying (permute for exactly two axes); x.T swaps the two axes of a matrix. NumPy arrays have a method of the same name that takes the whole new order instead, like permute: img.transpose(2, 0, 1) turns an (H, W, C) array into (C, H, W).</li><li><code>x.contiguous()</code> &nbsp; Returns a tensor with the same values laid out in memory in the order of its shape: a copy if x is not contiguous (after permute or transpose), x itself if it already is. Needed before .view() and by some libraries that read the raw memory.</li><li><code>x.item()</code> &nbsp; Returns the value of a one-element tensor as a plain Python number (float, int or bool). Used to print or store a loss: loss.item(). The number is no longer tracked by autograd and no longer on the GPU.</li><li><code>x.permute(*dims)</code> &nbsp; Reorders the dimensions: the argument lists the old positions in their new order. No data is copied. Matplotlib and image files use (H, W, C); PyTorch layers use (C, H, W); permute converts.</li></ul></details>

In [ ]:
ones = np.ones((6, 6))
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, k in zip(axes[:2], [3, 4]):
    o = convops.transposed_conv2d(ones, np.ones((k, k)), 2)
    ax.imshow(o, cmap="YlOrRd", vmin=0, vmax=6); viz.annotate_grid(ax, o, fontsize=6)
    ax.set_title(f"stride 2, kernel {k}: overlap count"); ax.axis("off")
# a real image upsampled x4 with two stride-2 transposed convs and FIXED smoothing kernels
small_img = torch.tensor(img[::8, ::8].transpose(2, 0, 1).copy())[None].float()
kernels = {"k=3 box [1 1 1]": [1.0, 1.0, 1.0], "k=4 bilinear [.25 .75 .75 .25]": [0.25, 0.75, 0.75, 0.25]}
for ax, (name, k1) in zip(axes[2:], kernels.items()):
    kk = torch.tensor(np.outer(k1, k1), dtype=torch.float32)
    up = small_img
    for _ in range(2):
        up = F.conv_transpose2d(up, kk.expand(3, 1, *kk.shape).contiguous(), stride=2, groups=3)
    up = up[0].permute(1, 2, 0).numpy()
    crop = up[100:160, 120:180]
    ax.imshow(np.clip(crop / up.mean() * small_img.mean().item(), 0, 1), interpolation="nearest")
    ax.set_title(f"Pet image x4 (crop), {name}", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

The same effect with **learned (random-init)** layers: three x2 stages applied to a
*constant* input. A perfect upsampler would return a constant image.

<details class="api-ref" data-api="nn.ConvTranspose2d,nn.Upsample,nn.PixelShuffle,Module.modules,nn.init.uniform_,nn.init.zeros_"><summary><b>Functions used here</b> (click to open): <code>nn.ConvTranspose2d</code> · <code>nn.Upsample</code> · <code>nn.PixelShuffle</code> · <code>Module.modules</code> · <code>nn.init.uniform_</code> · <code>nn.init.zeros_</code></summary>
<p><b><code>nn.ConvTranspose2d(in_ch, out_ch, kernel_size, stride=1, padding=0, output_padding=0)</code></b><br>A learnable upsampling layer: the transposed convolution of F.conv_transpose2d (each input pixel paints a scaled copy of the kernel into the output; overlaps add up). Output side = (H - 1) * stride - 2 * padding + kernel_size + output_padding. kernel_size=2, stride=2 doubles H and W with no overlap at all: the upsampler of the U-Net decoder.</p><pre>&gt;&gt;&gt; x = torch.zeros(1, 8, 16, 16)
&gt;&gt;&gt; nn.ConvTranspose2d(8, 4, kernel_size=2, stride=2)(x).shape       # (16-1)*2 + 2 = 32
torch.Size([1, 4, 32, 32])
&gt;&gt;&gt; nn.ConvTranspose2d(8, 4, 3, stride=2, padding=1)(x).shape        # (16-1)*2 - 2 + 3 = 31
torch.Size([1, 4, 31, 31])
&gt;&gt;&gt; nn.ConvTranspose2d(8, 4, 3, stride=2, padding=1, output_padding=1)(x).shape
torch.Size([1, 4, 32, 32])
&gt;&gt;&gt; nn.ConvTranspose2d(8, 4, 4, stride=2, padding=1)(x).shape        # (16-1)*2 - 2 + 4 = 32
torch.Size([1, 4, 32, 32])</pre><p><i>Watch out:</i> When kernel_size is not divisible by stride (3 with stride 2) neighbouring output pixels receive different numbers of overlapping stamps: the checkerboard artefact. Use kernel 2 or 4 with stride 2, or nn.Upsample followed by nn.Conv2d.</p><p>See also: <code>F.conv_transpose2d</code>, <code>nn.Upsample</code>, <code>nn.PixelShuffle</code></p><p><b><code>nn.Upsample(scale_factor=None, size=None, mode=&quot;nearest&quot;, align_corners=None)</code></b><br>A layer without weights that resizes (N, C, H, W) feature maps by a factor, or to a fixed size, by calling F.interpolate. nn.Upsample(scale_factor=2, mode=&quot;bilinear&quot;) followed by a 3x3 nn.Conv2d is the artefact-free alternative to a transposed convolution.</p><pre>&gt;&gt;&gt; x = torch.tensor([[[[1., 2.],
...                     [3., 4.]]]])
&gt;&gt;&gt; nn.Upsample(scale_factor=2, mode=&quot;nearest&quot;)(x)       # every pixel copied into a 2x2 block
tensor([[[[1., 1., 2., 2.],
          [1., 1., 2., 2.],
          [3., 3., 4., 4.],
          [3., 3., 4., 4.]]]])
&gt;&gt;&gt; nn.Upsample(scale_factor=2, mode=&quot;bilinear&quot;)(x)[0, 0, 0]      # first row: blends neighbours
tensor([1.0000, 1.2500, 1.7500, 2.0000])</pre><p><i>Watch out:</i> It has no parameters: on its own it can only copy or blend pixels, so a learned layer must follow it.</p><p>See also: <code>F.interpolate</code>, <code>nn.ConvTranspose2d</code></p><p><b><code>nn.PixelShuffle(r)</code></b><br>Rearranges (N, C * r * r, H, W) into (N, C, H * r, W * r): each group of r * r channels becomes the r x r block of pixels at the same position. It has no weights; the convolution in front of it predicts the r * r sub-pixels at low resolution (&quot;sub-pixel convolution&quot;).</p><pre>&gt;&gt;&gt; x = torch.tensor([1., 2., 3., 4.]).reshape(1, 4, 1, 1)   # 4 channels, 1 pixel
&gt;&gt;&gt; nn.PixelShuffle(2)(x)                                    # 1 channel, 2x2 pixels
tensor([[[[1., 2.],
          [3., 4.]]]])
&gt;&gt;&gt; nn.PixelShuffle(2)(torch.zeros(1, 12, 5, 5)).shape      # 12 = 3 channels x 2 x 2
torch.Size([1, 3, 10, 10])</pre><p><i>Watch out:</i> The number of channels must be divisible by r * r. nn.PixelUnshuffle does the reverse.</p><p>See also: <code>nn.ConvTranspose2d</code>, <code>nn.Upsample</code></p><p><b><code>model.modules()</code></b><br>Iterates over the model itself and every layer inside it, at any depth (layers inside Sequentials inside blocks...). The usual way to visit every layer of one type, e.g. to re-initialise the weights of all Conv2d layers.</p><pre>&gt;&gt;&gt; net = nn.Sequential(nn.Conv2d(1, 4, 3), nn.Sequential(nn.ReLU(), nn.Conv2d(4, 2, 1)))
&gt;&gt;&gt; [type(m).__name__ for m in net.modules()]
[&#x27;Sequential&#x27;, &#x27;Conv2d&#x27;, &#x27;Sequential&#x27;, &#x27;ReLU&#x27;, &#x27;Conv2d&#x27;]
&gt;&gt;&gt; sum(isinstance(m, nn.Conv2d) for m in net.modules())
2</pre><p><i>Watch out:</i> The first item is the model itself. children() gives only the direct sub-modules, one level down.</p><p>See also: <code>Module.children</code></p><p><b><code>nn.init.uniform_(tensor, a=0.0, b=1.0)</code></b><br>Overwrites a tensor in place with random numbers drawn uniformly between a and b, and returns it. Used on a layer&#x27;s weight to replace the default initialisation; D3-01 sets all weights positive so that the overlapping stamps of a transposed convolution become visible.</p><pre>&gt;&gt;&gt; _ = torch.manual_seed(0)
&gt;&gt;&gt; w = torch.empty(2, 3)
&gt;&gt;&gt; _ = nn.init.uniform_(w, 0, 1)                # w itself now holds the numbers
&gt;&gt;&gt; w
tensor([[0.4963, 0.7682, 0.0885],
        [0.1320, 0.3074, 0.6341]])
&gt;&gt;&gt; conv = nn.Conv2d(8, 8, 3)
&gt;&gt;&gt; bool(nn.init.uniform_(conv.weight, 0, 1).min() &gt;= 0)
True</pre><p>See also: <code>nn.init.zeros_</code>, <code>nn.init.constant_</code></p><p><b><code>nn.init.zeros_(tensor)</code></b><br>Fills a tensor with zeros in place, and returns it. Typically used on a layer&#x27;s bias after initialising its weights: nn.init.zeros_(conv.bias).</p><pre>&gt;&gt;&gt; lin = nn.Linear(3, 2)
&gt;&gt;&gt; nn.init.zeros_(lin.bias)
Parameter containing:
tensor([0., 0.], requires_grad=True)</pre><p>See also: <code>nn.init.uniform_</code>, <code>nn.init.constant_</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="nn.Sequential,nn.Conv2d,torch.ones,torch.no_grad,Tensor.std"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>nn.Sequential</code> · <code>nn.Conv2d</code> · <code>torch.ones</code> · <code>torch.no_grad</code> · <code>Tensor.std</code></summary>
<ul><li><code>nn.Sequential(layer1, layer2, ...)</code> &nbsp; A model made of layers applied one after the other, the output of each being the input of the next. Saves writing a class when forward() would only chain layers. The layers can be read back by position: model[0].</li><li><code>nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, bias=True)</code> &nbsp; A convolution layer: out_channels learned kernels of size (in_channels, k, k), each slid over the input exactly like this notebook&#x27;s cross-correlation (no kernel flip). Input (B, C_in, H, W) -&gt; output (B, C_out, H_out, W_out), with H_out from the output-size formula. The weight has shape (C_out, C_in, k, k), the bias (C_out,).</li><li><code>torch.ones(*size, dtype=None, device=None)</code> &nbsp; A new tensor of the given shape filled with 1 (float32 unless dtype says otherwise).</li><li><code>with torch.no_grad(): ...</code> &nbsp; A with block in which autograd records nothing: results have requires_grad=False, and computing them takes less memory and time. Use it when you only evaluate a model (testing, plotting its predictions), never around the training step itself.</li><li><code>x.std(dim=None, correction=1)</code> &nbsp; The standard deviation of all elements, or along dim. PyTorch divides by N - 1 by default (the sample standard deviation), while NumPy&#x27;s arr.std(axis) divides by N. Used for the per-channel statistics of a normalisation.</li></ul></details>

In [ ]:
torch.manual_seed(0)
def stack(kind):
    layers = []
    for _ in range(3):
        if kind == "convT k3 s2":   layers.append(nn.ConvTranspose2d(8, 8, 3, stride=2, padding=1, output_padding=1))
        elif kind == "convT k4 s2": layers.append(nn.ConvTranspose2d(8, 8, 4, stride=2, padding=1))
        elif kind == "bilinear + conv3x3": layers += [nn.Upsample(scale_factor=2, mode="bilinear"), nn.Conv2d(8, 8, 3, padding=1, padding_mode="replicate")]
        else: layers += [nn.Conv2d(8, 32, 3, padding=1, padding_mode="replicate"), nn.PixelShuffle(2)]
    net = nn.Sequential(*layers)
    for m in net.modules():
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.uniform_(m.weight, 0, 1); nn.init.zeros_(m.bias)   # positive weights make overlap visible
    return net

x_const = torch.ones(1, 8, 8, 8)
kinds = ["convT k3 s2", "convT k4 s2", "bilinear + conv3x3", "conv + pixel shuffle"]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.9))
for ax, kind in zip(axes, kinds):
    with torch.no_grad():
        y = stack(kind)(x_const)[0, 0].numpy()
    c = y[8:-8, 8:-8]                                       # ignore the border
    ax.imshow(c / c.mean(), cmap="RdBu_r", vmin=0.8, vmax=1.2); ax.axis("off")
    ax.set_title(f"{kind}\nstd/mean in the interior: {c.std() / c.mean():.3f}")
plt.suptitle("constant input, three random-init x2 upsampling stages (64 x 64 output, interior shown;\n"
             "colour = value / mean on a fixed 0.8-1.2 scale: a perfect upsampler is uniformly white)", y=1.06)
plt.tight_layout(); plt.show()

Transposed conv with k=3 shows a strong grid; k=4 overlaps evenly but its four
sub-kernels are different random numbers, so a faint ripple remains (and can grow
during training); resize + conv is exactly flat. Pixel shuffle with independently
initialised sub-kernels also leaves a periodic ripple at init - which is why it is used
with *ICNR* initialisation (all $r^2$ sub-kernels start identical). Modern decoders: **bilinear + conv** (DeepLab,
SegFormer, FPN) or **k=2, s=2 transposed conv** (U-Net: no overlap at all).

### Bilinear upsampling vs the library

`convops.bilinear_upsample` is a 15-line reference; check it against `F.interpolate` and
compare the two learned upsamplers on one feature map.

In [ ]:
fmap = torch.tensor(img[::16, ::16].mean(-1), dtype=torch.float32)
ref = F.interpolate(fmap[None, None], scale_factor=2, mode="bilinear", align_corners=False)[0, 0].numpy()
print("convops.bilinear_upsample == F.interpolate(bilinear, align_corners=False):",
      np.allclose(convops.bilinear_upsample(fmap.numpy(), 2), ref, atol=1e-5))

torch.manual_seed(1)
up_t = nn.ConvTranspose2d(1, 1, 3, stride=2, padding=1, output_padding=1)
up_b = nn.Sequential(nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False), nn.Conv2d(1, 1, 3, padding=1))
with torch.no_grad():
    outs = {"input (1/16 of the Pet image)": fmap.numpy(), "nn.ConvTranspose2d(k3, s2)": up_t(fmap[None, None])[0, 0].numpy(),
            "nn.Upsample(bilinear) + Conv2d": up_b(fmap[None, None])[0, 0].numpy()}
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (k, v) in zip(axes, outs.items()):
    ax.imshow(v, cmap="gray"); ax.set_title(f"{k}\n{v.shape[0]}x{v.shape[1]}", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

### Pixel shuffle (sub-pixel convolution)

Predict $C \cdot r^2$ channels at low resolution, then **rearrange**: channel group
$(c, a, b)$ becomes the pixel at offset $(a, b)$ inside each $r \times r$ output cell. No
zeros are inserted and every output pixel is computed at low resolution, which makes it
cheap (ESPCN super-resolution, many real-time decoders).

<details class="api-ref" data-api="np.floor"><summary><b>Functions used here</b> (click to open): <code>np.floor</code></summary>
<p><b><code>np.floor(x)</code></b><br>Rounds every element down to the nearest whole number (towards minus infinity) and keeps the float dtype; add .astype(int) to use the result as an index.</p><pre>&gt;&gt;&gt; np.floor(np.array([1.7, 2.0, -0.5]))
array([ 1.,  2., -1.])
&gt;&gt;&gt; np.floor(np.array([3.4, 0.9])).astype(int)
array([3, 0])</pre><p><i>Watch out:</i> Negative numbers go down: floor(-0.5) is -1, while int(-0.5) and .astype(int) cut towards 0 and give 0.</p><p>See also: <code>torch.round</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="np.stack,np.full,torch.randn"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.stack</code> · <code>np.full</code> · <code>torch.randn</code></summary>
<ul><li><code>np.stack(arrays, axis=0)</code> &nbsp; Joins arrays of the same shape along a NEW axis: k arrays of shape (N,) give (k, N) with axis=0, or (N, k) with axis=-1. The box conversions use it to put four (N,) coordinate columns back together into one (N, 4) array.</li><li><code>np.full(shape, value)</code> &nbsp; An array of the given shape (a number or a tuple) with every element set to value.</li><li><code>torch.randn(*size, generator=None)</code> &nbsp; A tensor of the given shape filled with random numbers from the standard normal distribution (mean 0, standard deviation 1); 0.4 * torch.randn(n) has standard deviation 0.4. The course uses it for noise and for random test inputs.</li></ul></details>

In [ ]:
r = 2
xps = np.stack([np.full((3, 3), c + 1.0) for c in range(4)])     # 4 channels = 1 output channel x r^2
xps += np.arange(9).reshape(3, 3) / 10                           # tag each position: channel.position
fig, axes = plt.subplots(1, 5, figsize=(14, 3), gridspec_kw=dict(width_ratios=[1, 1, 1, 1, 2]))
cols = ["Blues", "Oranges", "Greens", "Reds"]
for c in range(4):
    axes[c].imshow(np.ones((3, 3)), cmap=cols[c], vmin=0, vmax=2); viz.annotate_grid(axes[c], xps[c], fmt="{:.1f}", fontsize=8)
    axes[c].set_title(f"channel {c} = offset {divmod(c, 2)}", fontsize=9); axes[c].set_xticks([]); axes[c].set_yticks([])
ps = seg.pixel_shuffle(xps, r)[0]
rgb = np.zeros(ps.shape + (3,))
for c, cm_ in enumerate(cols):
    rgb[np.floor(ps).astype(int) - 1 == c] = plt.get_cmap(cm_)(0.5)[:3]
axes[4].imshow(rgb); viz.annotate_grid(axes[4], ps, fmt="{:.1f}", fontsize=8)
axes[4].set_title("pixel shuffle -> 1 x 6 x 6"); axes[4].set_xticks([]); axes[4].set_yticks([])
plt.tight_layout(); plt.show()

### Exercise 6 · `pixel_shuffle`

`(C*r*r, H, W) -> (C, H*r, W*r)`: one `reshape`, one `transpose`, one `reshape`. The
order of the axes in the transpose is the whole exercise.

<details class="api-ref" data-api="Tensor.transpose,Tensor.reshape"><summary><b>Functions you will need (met before)</b> (click to open): <code>Tensor.transpose</code> · <code>Tensor.reshape</code></summary>
<ul><li><code>x.transpose(dim0, dim1)</code> &nbsp; Swaps two dimensions of a tensor without copying (permute for exactly two axes); x.T swaps the two axes of a matrix. NumPy arrays have a method of the same name that takes the whole new order instead, like permute: img.transpose(2, 0, 1) turns an (H, W, C) array into (C, H, W).</li><li><code>x.reshape(*shape)</code> &nbsp; Returns the same elements in a new shape, read in row order (the last dimension changes fastest). One size may be -1: it is computed from the others. NumPy arrays have the same method.</li></ul></details>

In [ ]:
def pixel_shuffle(x, r):
    '''x (C*r*r, H, W) NumPy array -> (C, H*r, W*r): channel (c, a, b) becomes the pixel at offset (a, b)
    inside every r x r output cell.'''
    C_r2, H, W = x.shape
    C = C_r2 // (r * r)
    # 1. YOUR CODE: split the channel axis into (c, a, b): x.reshape(C, r, r, H, W)
    # 2. YOUR CODE: transpose the axes to the order (C, H, a, W, b), so that a sits next to H and b next to W
    # 3. YOUR CODE: merge (H, a) into H*r and (W, b) into W*r with one reshape; return the result
    raise NotImplementedError

tests.check_pixel_shuffle(pixel_shuffle)

## 5 · Context without losing resolution: dilated convolution and ASPP

A **dilated** (atrous, "with holes") 3x3 convolution with rate $d$ samples the input at
offsets $\{-d, 0, d\}^2$: same 9 weights, receptive field $2d + 1$. Stacking rates
$1, 2, 4$ grows the receptive field *exponentially* while the feature map keeps its
resolution - DeepLab removes the last downsampling stages of a ResNet and dilates the
rest instead.

### Exercise 7 · `dilate_kernel`

Build the equivalent dense kernel: insert `d - 1` zeros between the taps. Size
$d(k - 1) + 1$.

In [ ]:
def dilate_kernel(k, dilation):
    '''k (kh, kw) -> the equivalent dense kernel (dilation*(kh-1) + 1, dilation*(kw-1) + 1),
    with dilation - 1 zeros between the taps.'''
    kh, kw = k.shape
    # 1. YOUR CODE: out_h and out_w, the size of the dense kernel: dilation * (kh - 1) + 1 and the same for kw
    # 2. YOUR CODE: out, zeros of that size, with the same dtype as k
    # 3. YOUR CODE: put the taps at every dilation-th row and column: out[::dilation, ::dilation] = k
    # 4. YOUR CODE: return out
    raise NotImplementedError

tests.check_dilate_kernel(dilate_kernel)

Which input pixels can influence the centre output after three 3x3 layers? Ask autograd:
the gradient of the centre output w.r.t. the input is non-zero exactly on the receptive
field. Equal rates (2, 2, 2) leave **holes** (the "gridding" problem), which is why
DeepLabv3 and HDC (hybrid dilated convolution, Wang et al. 2018) use rates with no common factor.

<details class="api-ref" data-api="np.nonzero"><summary><b>Functions used here</b> (click to open): <code>np.nonzero</code></summary>
<p><b><code>rows, cols = np.nonzero(a)</code></b><br>The indices of the non-zero (or True) elements, as one array per dimension: for a 2-D mask, the row indices and the column indices. Same as np.where(a) with one argument. ys.min(), xs.max() etc. then give the mask&#x27;s bounding box.</p><pre>&gt;&gt;&gt; m = np.array([[0, 0, 0],
...               [0, 1, 1],
...               [0, 1, 0]])
&gt;&gt;&gt; ys, xs = np.nonzero(m)                       # rows first, then columns
&gt;&gt;&gt; ys, xs
(array([1, 1, 2]), array([1, 2, 1]))
&gt;&gt;&gt; int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())    # box x1, y1, x2, y2
(1, 1, 2, 2)</pre><p><i>Watch out:</i> The rows come first: ys, xs = np.nonzero(mask), not xs, ys.</p><p>See also: <code>np.where</code>, <code>torch.where</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="nn.init.constant_"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>nn.init.constant_</code></summary>
<ul><li><code>nn.init.constant_(tensor, value)</code> &nbsp; Fills a parameter with one value, in place (the trailing _). Detectors set the bias of the last classification conv to -log((1 - pi) / pi) with pi = 0.01, so that at the start every anchor predicts &quot;1% object&quot; and the many background anchors do not blow up the first losses.</li></ul></details>

In [ ]:
def rf_map(dilations, n=33):
    layers = []
    for d in dilations:
        c = nn.Conv2d(1, 1, 3, padding=d, dilation=d, bias=False); nn.init.constant_(c.weight, 1.0); layers.append(c)
    x = torch.zeros(1, 1, n, n, requires_grad=True)
    nn.Sequential(*layers)(x)[0, 0, n // 2, n // 2].backward()
    return x.grad[0, 0].numpy()

configs = [(1, 1, 1), (1, 2, 4), (2, 2, 2), (1, 2, 4, 8)]
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, d in zip(axes, configs):
    g = rf_map(d)
    ys_, xs_ = np.nonzero(g)
    ax.imshow(np.log1p(g), cmap="magma"); ax.set_xticks([]); ax.set_yticks([])
    holes = (g[ys_.min():ys_.max() + 1, xs_.min():xs_.max() + 1] == 0).sum()
    ax.set_title(f"dilations {d}\nRF {ys_.max() - ys_.min() + 1}x{xs_.max() - xs_.min() + 1}, holes: {holes}")
plt.suptitle("receptive field of the centre pixel (colour = number of paths, log)", y=1.02)
plt.tight_layout(); plt.show()

**ASPP** (Atrous Spatial Pyramid Pooling, DeepLabv3) runs several dilated 3x3 convs in
*parallel* on the same feature map (rates 6, 12, 18 at output stride 16, i.e. a feature map
16x smaller than the input), plus a 1x1
conv and a global-average-pooled branch, and concatenates them: multi-scale context in
one layer. Each branch samples the map like this:

<details class="api-ref" data-api="nn.ModuleList,torch.cat"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>nn.ModuleList</code> · <code>torch.cat</code></summary>
<ul><li><code>nn.ModuleList([module, ...])</code> &nbsp; A list of layers that the model knows about: their parameters appear in model.parameters() (so the optimizer trains them) and move with model.to(device). Use it for a variable number of layers that your forward() calls in its own way, e.g. one lateral conv per pyramid level; unlike nn.Sequential it has no forward of its own.</li><li><code>torch.cat(tensors, dim=0)</code> &nbsp; Joins a list of tensors along an existing dimension dim; all other dimensions must match. dim=0 appends samples (collect per-batch results into one tensor); dim=1 on (N, C, H, W) tensors puts channels side by side, as an Inception block does with its branches.</li></ul></details>

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 6.2))
n = 45; c = n // 2
ax.imshow(np.zeros((n, n)), cmap="Greys", vmin=0, vmax=1)
ax.add_patch(Rectangle((-0.5, -0.5), n, n, fc="#f3f3f3", ec="none"))
for rate, col in [(1, "#4C78A8"), (6, "#F58518"), (12, "#54A24B"), (18, "#E45756")]:
    for dy in (-1, 0, 1):
        for dx in (-1, 0, 1):
            ax.add_patch(Rectangle((c + dx * rate - 0.5, c + dy * rate - 0.5), 1, 1, fc=col, ec="k", lw=0.4))
    ax.plot([], [], "s", color=col, label=f"3x3, rate {rate}")
ax.plot([], [], "s", color="#BAB0AC", label="+ 1x1 conv + image pooling")
ax.legend(loc="upper left", fontsize=8, bbox_to_anchor=(1.01, 1)); ax.set_xticks([]); ax.set_yticks([])
ax.set_title("ASPP: four branches sample the same feature map at four scales")
plt.tight_layout(); plt.show()
aspp = nn.ModuleList()                                  # four parallel branches, one per rate
for rate in (1, 6, 12, 18):
    aspp.append(nn.Conv2d(256, 64, 3, padding=rate, dilation=rate))   # padding = rate keeps the size
f = torch.randn(1, 256, 32, 32)
branch_outputs = []
branch_shapes = []
for branch in aspp:
    branch_out = branch(f)                              # (1, 64, 32, 32) for every rate
    branch_outputs.append(branch_out)
    branch_shapes.append(tuple(branch_out.shape))
concat = torch.cat(branch_outputs, 1)                   # (1, 4 * 64, 32, 32): all scales side by side
print("ASPP branch outputs:", branch_shapes, "-> concat:", tuple(concat.shape))

## 6 · The ancestor: a convolutional autoencoder

An **encoder-decoder** compresses the image into a small bottleneck and reconstructs it.
Replace "reconstruct the image" by "predict the mask" and you have a segmentation network.
We train a 3-level conv autoencoder on FashionMNIST for one epoch
(28 -> 14 -> 7 -> **4x4x8 bottleneck** -> 7 -> 14 -> 28, transposed convs on the way up),
and an identical one with **one skip connection** that hands the decoder the 14x14
encoder features.

<details class="api-ref" data-api="F.mse_loss,scipy.ndimage.sobel,np.hypot,np.corrcoef"><summary><b>Functions used here</b> (click to open): <code>F.mse_loss</code> · <code>scipy.ndimage.sobel</code> · <code>np.hypot</code> · <code>np.corrcoef</code></summary>
<p><b><code>F.mse_loss(pred, target)</code></b><br>Mean squared error: the mean of (pred - target)**2 over all elements of two tensors of the same shape. The function form of nn.MSELoss; D3-01 uses it to score how well the autoencoder reconstructs the test images.</p><pre>&gt;&gt;&gt; F.mse_loss(torch.tensor([1., 2., 3.]), torch.tensor([1., 0., 6.]))   # (0 + 4 + 9) / 3
tensor(4.3333)</pre><p><i>Watch out:</i> Shapes must match exactly: pred (N, 1) with target (N,) is broadcast to (N, N) and gives a wrong number, with only a warning.</p><p>See also: <code>nn.MSELoss</code></p><p><b><code>ndimage.sobel(image, axis=-1)</code></b><br>The Sobel derivative of a 2-D image along one axis: large where the brightness changes along that axis (axis=1: left-right change, i.e. vertical edges; axis=0: top-bottom change). D3-01 measures edge strength as np.hypot(sobel(im, 0), sobel(im, 1)).</p><pre>&gt;&gt;&gt; from scipy import ndimage
&gt;&gt;&gt; im = np.zeros((4, 4)); im[:, 2:] = 1.0       # dark left half, bright right half
&gt;&gt;&gt; ndimage.sobel(im, axis=1)                    # change from left to right: the edge
array([[0., 4., 4., 0.],
       [0., 4., 4., 0.],
       [0., 4., 4., 0.],
       [0., 4., 4., 0.]])
&gt;&gt;&gt; float(np.abs(ndimage.sobel(im, axis=0)).max())      # nothing changes from top to bottom
0.0</pre><p><i>Watch out:</i> The result has the input&#x27;s dtype: on a uint8 image the negative and large values wrap around (255 * 4 becomes 252). Convert to float first.</p><p>See also: <code>np.hypot</code></p><p><b><code>np.hypot(a, b)</code></b><br>Element-wise sqrt(a**2 + b**2): the length of the vector (a, b). Applied to the horizontal and vertical Sobel derivatives of an image, it gives the edge strength at every pixel.</p><pre>&gt;&gt;&gt; np.hypot(np.array([3., 0., 5.]), np.array([4., 2., 12.]))
array([ 5.,  2., 13.])</pre><p>See also: <code>scipy.ndimage.sobel</code></p><p><b><code>np.corrcoef(x, y)</code></b><br>Pearson correlation between two 1-D arrays of the same length: +1 when one grows exactly in step with the other (on a straight line), 0 when they are unrelated, -1 when one falls as the other rises. Returns the 2x2 matrix [[1, r], [r, 1]]; the number you want is [0, 1].</p><pre>&gt;&gt;&gt; x = np.array([1., 2., 3., 4.])
&gt;&gt;&gt; round(float(np.corrcoef(x, 2 * x + 1)[0, 1]), 3)      # a straight line
1.0
&gt;&gt;&gt; round(float(np.corrcoef(x, [4., 3., 2., 1.])[0, 1]), 3)
-1.0
&gt;&gt;&gt; round(float(np.corrcoef(x, [1., 3., 2., 4.])[0, 1]), 3)
0.8</pre><p><i>Watch out:</i> Flatten images first (a.ravel()): with 2-D inputs every row counts as a separate variable.</p><p>See also: <code>Tensor.ravel</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="torchkit.get_fashion_mnist,torch.utils.data.DataLoader,torch.optim.AdamW,Module.parameters,Tensor.to,F.binary_cross_entropy_with_logits,Optimizer.zero_grad,Optimizer.step,Module.eval,torch.sigmoid,Tensor.cpu,Tensor.ravel"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torchkit.get_fashion_mnist</code> · <code>torch.utils.data.DataLoader</code> · <code>torch.optim.AdamW</code> · <code>Module.parameters</code> · <code>Tensor.to</code> · <code>F.binary_cross_entropy_with_logits</code> · <code>Optimizer.zero_grad</code> · <code>Optimizer.step</code> · <code>Module.eval</code> · <code>torch.sigmoid</code> · <code>Tensor.cpu</code> · <code>Tensor.ravel</code></summary>
<ul><li><code>train, test, classes = torchkit.get_fashion_mnist(normalize=True, as_tensors=False)</code> &nbsp; Course helper. Returns the 60 000 training and 10 000 test images of FashionMNIST and the list of its 10 class names, downloading the data into data/ on first use. An item ds[i] is (image, label): a float tensor (1, 28, 28), normalised with mean 0.286 and std 0.353 unless normalize=False (then in [0, 1]), and the class index. as_tensors=True holds everything in memory as a TensorDataset, which is much faster on a CPU.</li><li><code>DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=None)</code> &nbsp; Turns a dataset (anything with len(ds) and ds[i]) into batches you can loop over: each step takes batch_size items and stacks them, so images (C, H, W) become one tensor (B, C, H, W) and integer labels a tensor (B,). shuffle=True draws a new random order every epoch (for training; evaluation does not need it).</li><li><code>torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)</code> &nbsp; Adam with decoupled weight decay, the default optimiser of modern training. Adam divides each parameter&#x27;s step by a running average of its own gradient size, so parameters move at a similar pace whatever the scale of their gradients; AdamW also shrinks every weight by the fraction lr * weight_decay at each step, which keeps the weights small and reduces overfitting. Used exactly like SGD.</li><li><code>model.parameters()</code> &nbsp; All learnable tensors of a model (the weights and biases of every layer inside it), one after the other. What you hand to an optimiser: torch.optim.SGD(model.parameters(), lr=0.1).</li><li><code>x.to(device) / x.to(dtype) / model.to(device)</code> &nbsp; Returns the tensor converted to another device (x.to(&quot;cuda&quot;)), another dtype (x.to(torch.float32)) or both; if nothing changes, x itself. On a model, model.to(device) moves all its weights and returns the model. The course moves every model and every batch to the device picked by torchkit.get_device.</li><li><code>F.binary_cross_entropy_with_logits(logits, targets, reduction=&#x27;mean&#x27;)</code> &nbsp; Binary cross-entropy for yes / no outputs, computed from raw logits z: per element -[t log sigmoid(z) + (1 - t) log(1 - sigmoid(z))], then the mean (reduction=&quot;none&quot; keeps one loss per element, &quot;sum&quot; adds them up). targets are floats 0. / 1. of the same shape. Objectness and focal loss use it.</li><li><code>optimizer.zero_grad()</code> &nbsp; Clears the .grad of every parameter the optimiser updates (sets it to None), so that the next backward() starts from zero. Call it in every training step, before loss.backward().</li><li><code>optimizer.step()</code> &nbsp; Updates every parameter the optimiser was given, using the gradients now in .grad; for plain SGD, w &lt;- w - lr * w.grad. Call it after loss.backward().</li><li><code>model.eval()</code> &nbsp; Puts the model and all its layers into evaluation mode: dropout is switched off and BatchNorm uses the running averages collected during training, so a prediction no longer depends on the rest of the batch. Returns the model, so get_model(...).eval() works in one line.</li><li><code>torch.sigmoid(x)</code> &nbsp; Squashes every value (a logit) into (0, 1): sigmoid(x) = 1 / (1 + exp(-x)). Turns an objectness logit into a probability; D2-03 and D2-04 also use it to keep predicted box coordinates between 0 and 1.</li><li><code>x.cpu()</code> &nbsp; Returns a copy of the tensor in main memory (the CPU), or the tensor itself if it is already there. Needed before .numpy() or plotting when the result was computed on a GPU: x.detach().cpu().numpy().</li><li><code>x.ravel()</code> &nbsp; Returns all values as one flat 1-D tensor, row after row: the same as x.flatten() or x.reshape(-1). NumPy arrays have the same method; D1-02 calls it on an array (after .numpy()) to get a flat list of values for a histogram.</li></ul></details>

In [ ]:
train_fm, test_fm, fm_classes = torchkit.get_fashion_mnist(normalize=False, as_tensors=True)
sub = train_fm                                                  # one epoch = 60k images (capped at 60 s below)
loader = torch.utils.data.DataLoader(sub, batch_size=128, shuffle=True)
x_test = test_fm.tensors[0][:512]

aes = {}                                                        # name -> trained autoencoder
curves = {}                                                     # name -> loss of every step
for name, skip in [("no skip", False), ("one skip (14x14)", True)]:
    torch.manual_seed(0)
    ae = day3.ConvAutoencoder(bottleneck_ch=8, skip=skip).to(device)
    opt = torch.optim.AdamW(ae.parameters(), lr=3e-3)
    t0 = time.time()
    losses = []
    for xb, _ in loader:
        xb = xb.to(device)
        # the target is the input itself: reconstruct every pixel
        loss = F.binary_cross_entropy_with_logits(ae(xb), xb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
        if time.time() - t0 > 60:                                # wall-clock cap for slow machines
            break
    aes[name] = ae.eval()
    curves[name] = losses
    with torch.no_grad():
        mse = F.mse_loss(torch.sigmoid(ae(x_test.to(device))), x_test.to(device)).item()
    print(f"{name:18s} {len(losses)} steps in {time.time() - t0:4.0f}s   test MSE {mse:.4f}")

In [ ]:
from scipy.ndimage import sobel
idx = [0, 1, 2, 5, 13, 16]
rec = {}                                                        # name -> reconstructions of the 6 images
with torch.no_grad():
    for name, ae in aes.items():
        logits = ae(x_test[idx].to(device))                     # (6, 1, 28, 28)
        rec[name] = torch.sigmoid(logits).cpu()
fig, axes = plt.subplots(4, len(idx), figsize=(1.9 * len(idx), 8.2))
for j, i in enumerate(idx):
    axes[0, j].imshow(x_test[i, 0], cmap="gray"); axes[0, j].set_title(fm_classes[int(test_fm.tensors[1][i])], fontsize=8)
    axes[1, j].imshow(rec["no skip"][j, 0], cmap="gray", vmin=0, vmax=1)
    err = (rec["no skip"][j, 0] - x_test[i, 0]).abs().numpy()
    axes[2, j].imshow(err, cmap="inferno", vmin=0, vmax=0.6)
    axes[3, j].imshow(rec["one skip (14x14)"][j, 0], cmap="gray", vmin=0, vmax=1)
for r_, lab in enumerate(["input", "no skip", "|error| (no skip)", "with skip"]):
    axes[r_, 0].set_ylabel(lab, fontsize=10)
for ax in axes.flat: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

with torch.no_grad():
    err_all = (torch.sigmoid(aes["no skip"](x_test.to(device))).cpu() - x_test).abs()[:, 0].numpy()
edge_maps = []                                                  # edge strength of every test image
for im in x_test[:, 0].numpy():                                 # im: (28, 28)
    edge_maps.append(np.hypot(sobel(im, 0), sobel(im, 1)))      # length of the image gradient
edges = np.stack(edge_maps)                                     # (512, 28, 28)
print(f"correlation between reconstruction error and edge strength: {np.corrcoef(err_all.ravel(), edges.ravel())[0, 1]:.2f}")

The errors of the plain autoencoder sit on **edges, stripes and logos**: exactly the
"where" information (precise positions of boundaries) that does not fit through a 4x4
bottleneck. The bottleneck keeps *what* the object is; it forgets *exactly where* its
outline runs. For segmentation the boundary is the whole job.

The skip connection hands the decoder the high-resolution encoder features directly, and
the reconstruction becomes sharp. (For an autoencoder that is cheating - it can copy
the input - but for segmentation it is precisely what we want: **semantics from the
bottleneck, location from the skips**.) That is the U-Net, next notebook.

## 7 · The Pet trimap target

Oxford-IIIT Pet trimaps use `1 = pet`, `2 = background`, `3 = border/uncertain`. We remap
to the convention "0 is background": `0 background, 1 pet, 2 border`
(`torchkit.get_pet_datasets(target="segmentation")` does this for you). Two common
choices for the *border* class: keep it as a third class (what we do, it is the
official benchmark), or **ignore** it in the loss (`ignore_index=2` in
`F.cross_entropy`) because it is genuinely ambiguous.

<details class="api-ref" data-api="torchkit.get_pet_datasets,torchkit.denormalize,Tensor.unique,Tensor.tolist,np.bincount,np.median,np.round"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torchkit.get_pet_datasets</code> · <code>torchkit.denormalize</code> · <code>Tensor.unique</code> · <code>Tensor.tolist</code> · <code>np.bincount</code> · <code>np.median</code> · <code>np.round</code></summary>
<ul><li><code>train, val, classes = torchkit.get_pet_datasets(size=224, target=&quot;category&quot;, subset=None)</code> &nbsp; Course helper. Returns (train, val, classes) for Oxford-IIIT Pet (run tools/download_data.py first); images are float (3, size, size), resized and normalised with the ImageNet mean and std. target chooses the label: &quot;category&quot; gives (image, breed index 0..36) and the 37 breed names; &quot;segmentation&quot; gives (image, int64 mask (H, W)) with 0 background, 1 pet, 2 border; &quot;detection&quot; gives (image, {&quot;boxes&quot;: (N, 4) as x1, y1, x2, y2 in pixels, &quot;labels&quot;: (N,)}) for the pet&#x27;s head. subset=N keeps a class-balanced N training (and N // 4 validation) images for quick experiments.</li><li><code>torchkit.denormalize(x, mean=IMAGENET_MEAN, std=IMAGENET_STD)</code> &nbsp; Course helper. Undoes the normalisation (x * std + mean, per channel) and returns a NumPy array ready for plt.imshow, clipped to [0, 1]: (C, H, W) becomes (H, W, C), a batch (B, C, H, W) becomes (B, H, W, C), a 1-channel image becomes (H, W). The default statistics are ImageNet&#x27;s (torchkit.IMAGENET_MEAN, IMAGENET_STD); for FashionMNIST pass torchkit.FASHION_MNIST_MEAN, FASHION_MNIST_STD.</li><li><code>x.unique(return_counts=False)</code> &nbsp; The distinct values of a tensor, sorted, as a 1-D tensor; with return_counts=True also how often each one occurs. Handy to check which class ids a segmentation mask contains.</li><li><code>x.tolist()</code> &nbsp; Converts a tensor into (nested) Python lists of plain Python numbers, for printing a few values readably. A 0-dimensional tensor gives a single number, like .item().</li><li><code>np.bincount(x)</code> &nbsp; Counts how often each non-negative integer occurs: result[k] = how many times k appears in x, for k = 0 ... max(x). D2-04 turns &quot;detections per image&quot; into a histogram with it.</li><li><code>np.median(a, axis=None)</code> &nbsp; The middle value of the sorted data (the mean of the two middle values for an even count). A few slow outliers, such as a run disturbed by another program, shift the mean a lot but the median hardly at all: the right summary for repeated timings.</li><li><code>np.round(a, decimals=0)</code> &nbsp; Rounds every element to the given number of decimals. D2-01 cleans up the IoU thresholds 0.5, 0.55, ..., 0.95 made by np.arange with it: the floating-point steps of arange are not exact.</li></ul></details>

In [ ]:
pet_tr, pet_va, pet_classes = torchkit.get_pet_datasets(size=128, target="segmentation", subset=(240, 60))
print(len(pet_tr), "train /", len(pet_va), "val images; classes:", pet_classes)
xb, yb = next(iter(torch.utils.data.DataLoader(pet_tr, batch_size=8, shuffle=True)))
print("batch:", tuple(xb.shape), xb.dtype, "| masks:", tuple(yb.shape), yb.dtype, "values", yb.unique().tolist())

fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for j in range(8):
    axes[0, j].imshow(torchkit.denormalize(xb[j])); axes[0, j].axis("off")
    axes[1, j].imshow(torchkit.denormalize(xb[j])); axes[1, j].imshow(day3.colorize(yb[j]), alpha=0.55); axes[1, j].axis("off")
fig.legend(handles=[Patch(color=day3.SEG_COLORS[i], label=c) for i, c in enumerate(pet_classes)],
           loc="upper center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, 1.06))
plt.tight_layout(); plt.show()

In [ ]:
counts = np.zeros(3)
for _, m in torch.utils.data.DataLoader(pet_tr, batch_size=60):
    counts += np.bincount(m.numpy().ravel(), minlength=3)
freq = counts / counts.sum()
w_inv = 1 / freq; w_inv /= w_inv.mean()
w_med = np.median(freq) / freq                                   # "median frequency balancing" (SegNet)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].bar(pet_classes, freq, color=day3.SEG_COLORS[:3]); axes[0].set_title("pixel frequency per class (240 train images)")
axes[0].set_ylim(0, freq.max() * 1.18)
for i, v in enumerate(freq): axes[0].text(i, v + 0.01, f"{v:.1%}", ha="center")
xw = np.arange(3)
axes[1].bar(xw - 0.2, w_inv, 0.4, label="inverse frequency (mean 1)"); axes[1].bar(xw + 0.2, w_med, 0.4, label="median frequency")
axes[1].set_xticks(xw); axes[1].set_xticklabels(pet_classes); axes[1].legend(fontsize=8); axes[1].set_title("class weights for the loss")
plt.tight_layout(); plt.show()
print("loss with class weights:  F.cross_entropy(logits, target, weight=torch.tensor(", np.round(w_med, 2).tolist(), "))")
print("pixel accuracy of 'always predict the most frequent class':", f"{freq.max():.1%}")

Background and pet are roughly balanced here, but the **border** class is a thin band:
a model that never predicts "border" still scores high pixel accuracy. Median-frequency
weights push the border class up several times. Keep this picture in mind when you read
the per-class IoU of the U-Net in D3-02: the border IoU is always the lowest.

## 8 · Summary

* Semantic = class per pixel; instance = class + id for countable *things*; panoptic = both, every pixel.
* Masks are stored as dense PNGs, polygons or RLE (column-major runs starting with zeros).
* Never report pixel accuracy alone: use per-class IoU and mIoU from the confusion matrix; Dice = 2 IoU / (1 + IoU).
* Mean BCE lets the background drown small objects; Dice (or BCE + Dice, or focal) rebalances the gradient.
* Resolution comes back with transposed convs (scatter-add; keep kernel divisible by stride), bilinear + conv, or pixel shuffle; dilation and ASPP add context without downsampling.
* An encoder-decoder without skips keeps *what* but loses *where*; skip connections put it back.

### Self-check (answers in the Day 3 quiz)
1. A model predicts "background" everywhere on an image where the object covers 5% of the pixels. Pixel accuracy? IoU of the object? mIoU over the two classes?
2. IoU = 0.6. What is the Dice score?
3. Output size of `ConvTranspose2d(k=4, s=2, p=1)` on a 16x16 input? Does it produce a checkerboard from overlap?
4. Receptive field of three stacked 3x3 convs with dilations 1, 2, 4?
5. Why does a plain autoencoder blur edges, and what does a skip connection change?

### Go deeper
* Long, Shelhamer & Darrell, *Fully Convolutional Networks for Semantic Segmentation* (CVPR 2015).
* Odena, Dumoulin & Olah, *Deconvolution and Checkerboard Artifacts* (Distill, 2016).
* Chen et al., *DeepLab* (2017) and *Rethinking Atrous Convolution* (DeepLabv3, ASPP).
* Shi et al., *Real-Time Single Image and Video Super-Resolution Using an Efficient Sub-Pixel CNN* (pixel shuffle, 2016).
* Milletari, Navab & Ahmadi, *V-Net* (2016): the Dice loss.
* COCO data format and `pycocotools` mask API (RLE).